# 🛰️ Synthetic AMF KPI Dataset Generator — v14 (weekend-aware diurnal)
### Google Colab Edition

> **3GPP TS 28.552 v19.6.0** compliant — 21 procedures · 9 KPI sections · per-slice (eMBB / mMTC / URLLC) CPU / Memory / Latency · sigmoid anomaly onset/recovery · SLA breach detection.

**New in v14 vs v13:** per-slice resource KPIs, sigmoid anomaly ramps, Jackson open-network throughput model, 2 new anomaly types (`slice_isolation_failure`, `amf_overload_cascade`), SLA breach flags.

---
**How to use:**
1. Run **Cell 1** once (checks/installs deps).
2. Use **Runtime → Run all** (`Ctrl+F9`) to run everything.
3. Edit the **plain parameter values** in Section 11, then re-run from there.
4. The final cell exports a **ZIP** (CSV + JSON + plots + metadata) and triggers download.

In [1]:
PUBLIC_DROP_COLUMNS = ['anomaly_sigmoid_w', 'anomaly_intensity', 'composite_load', 'rho']

def build_public_release(df):
    drop_cols = [c for c in PUBLIC_DROP_COLUMNS if c in df.columns]
    return df.drop(columns=drop_cols).copy(), drop_cols



---
## Cell 1 — Install / Check Dependencies
Run once per Colab session.

In [2]:
# Uncomment the line below ONLY if a package is missing:
# !pip install -q scipy matplotlib pandas numpy ipywidgets

import importlib
for pkg in ['scipy', 'matplotlib', 'pandas', 'numpy', 'ipywidgets']:
    assert importlib.util.find_spec(pkg), f'{pkg} not found – uncomment the pip line above'
print('✓ All dependencies present.')


✓ All dependencies present.


In [3]:
# -*- coding: utf-8 -*-
"""
=============================================================================
Synthetic AMF KPI Dataset Generator – v14
=============================================================================
Authors : Research Team
Version : 14.0  (Enhanced Computational-Cost Model, Per-Slice Resources,
                  Gradual Anomaly Onset, Extended 3GPP Procedure Coverage)

IMPROVEMENTS OVER v13
─────────────────────
Statistical / Mathematical:
  ✔ Per-slice CPU model   – separate μ_CPU(slice, proc) from published
                            instruction-count measurements (Chiha et al. 2020)
  ✔ Per-slice memory model– UE-context size differs by slice (mMTC lightweight
                            vs URLLC with additional QoS state)
  ✔ Per-slice latency SLAs– URLLC targets <1 ms user-plane, eMBB <100 ms,
                            mMTC relaxed; modelled via slice-dependent σ_lognorm
  ✔ Gradual anomaly onset – anomalies ramp up / recover with a sigmoid envelope
                            instead of a hard step, making ML detection harder
                            and more realistic
  ✔ Composite load index  – entropy-based load diversity metric across slices
  ✔ Jackson-network model – multi-class open queueing network for N1/N2 msg flow
  ✔ Resource efficiency   – CPU cycles/message & memory bytes/UE per slice

Per-Slice Resource KPIs (new output columns):
  RES.CPU_eMBB_%           RES.CPU_mMTC_%          RES.CPU_URLLC_%
  RES.Mem_eMBB_MB          RES.Mem_mMTC_MB         RES.Mem_URLLC_MB
  RES.Lat_eMBB_ms          RES.Lat_mMTC_ms         RES.Lat_URLLC_ms
  RES.CpuCyclesPerMsg      RES.MemBytesPerUE
  RES.SliceLoadEntropy

Anomaly Engine:
  ✔ Sigmoid onset / recovery (configurable ramp_h parameter per scenario)
  ✔ Two new scenarios: slice_isolation_failure, amf_overload_cascade
  ✔ Intensity now supports float 0–1 "custom" values in addition to
    mild / moderate / severe labels

3GPP References:
  TS 28.552 §5.2   – AMF KPI definitions (v19.6.0)
  TS 23.501 §5     – Network slicing architecture
  TS 23.502 §4.2   – 5GC procedures and message flows
  TR 21.866        – Energy efficiency study item

Statistical References:
  Norros (1994)         – Self-similar fGn traffic model
  Chiha et al. (2020)   – AMF hardware dimensioning
  Papagiannaki (2003)   – GARCH volatility clustering
  Kleinrock (1975)      – M/M/c queueing theory
  Baskett et al. (1975) – BCMP / Jackson network theorem
  Shafiq et al. (2012)  – Mobile traffic diurnal patterns
=============================================================================
"""

# ── stdlib ─────────────────────────────────────────────────────────────────
import json
import math
import os
import warnings
import zipfile
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Tuple, Union

# Suppress only known benign warnings from third-party libraries
warnings.filterwarnings("ignore", category=RuntimeWarning, module="scipy")
warnings.filterwarnings("ignore", category=FutureWarning, module="pandas")

# ── third-party ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy.stats import nbinom, lognorm, norm, poisson, expon
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap


---
## Section 1 — Global Configuration (Event Catalogue · Slice Parameters · Anomaly Tables)

---
## Released Benchmark Configuration

> **Paper §VI, Table III** — The released benchmark uses:
> - `DURATION_HOURS = 1440` → **60 days** / 5,760 rows at 15-min granularity
> - `AMF_INSTANCES = 1` → single AMF instance (`AMF_00`)
> - `RANDOM_SEED = 42` → full reproducibility
>
> For ablation studies or rapid iteration, change `DURATION_HOURS` to `336` (14 days).
> The framework supports `AMF_INSTANCES` up to 5, but only `N=1` was validated for the released artifact.


In [4]:
RANDOM_SEED    = 42
DURATION_HOURS = 1440         # 60 days — released benchmark
# For ablation / rapid iteration use DURATION_HOURS=336 (14 days)
STEP_MIN       = 15           # 15-minute PM granularity (3GPP standard)
AMF_INSTANCES  = 1            # released benchmark (framework supports N≥1)
NUM_UES        = 100_000
VCPUS_PER_AMF  = 8
MEM_MAX_MB     = 8192.0       # 8 GB RAM per AMF VNF instance
HURST_EXPONENT = 0.75         # long-range dependence (self-similar traffic)

OUTPUT_DIR     = "/tmp/amf_dataset_v14"

# ─── 3GPP procedure event catalogue ────────────────────────────────────────
# bh_rate : mean events per UE per busy hour (calibrated from operational AMF
#            measurements and 3GPP TR 23.700-81 reference load models)
# msgs    : N1+N2 messages per procedure (3GPP TS 23.502 call flow message counts)
# nb_k    : Negative-Binomial dispersion parameter k  (larger k → Poisson limit)
# cpu_w   : CPU weight relative to Service Request = 1.0
#           Derived from Table 4 of Chiha et al. (2020): instruction counts
#           measured on a commercial AMF:
#             ServiceRequest  ≈ 3 580 k instr  → reference 1.00
#             ServiceRelease  ≈ 3 200 k instr  → 0.89
#             XnHandover      ≈ 2 140 k instr  → 0.60
#             InitReg         ≈ 4 200 k instr  → 1.17 (auth + key derivation)
# mem_ctx : UE-context memory footprint per procedure (MB per active context)
#           Base: ~5 KB NAS state; +2 KB security context per auth proc.
EVENT_DATA: Dict[str, Dict] = {
    "Initial Registration":      {"bh_rate": 0.625,  "msgs": 12, "nb_k": 60, "cpu_w": 1.17, "mem_ctx": 0.007},
    "Deregistration":            {"bh_rate": 0.208,  "msgs":  4, "nb_k": 50, "cpu_w": 0.45, "mem_ctx": 0.000},
    "Inter-AMF Mobility Reg":    {"bh_rate": 0.101,  "msgs":  8, "nb_k": 40, "cpu_w": 0.89, "mem_ctx": 0.005},
    "Intra-AMF Mobility Reg":    {"bh_rate": 1.575,  "msgs":  4, "nb_k": 70, "cpu_w": 0.60, "mem_ctx": 0.004},
    "Periodic Registration":     {"bh_rate": 0.5103, "msgs":  4, "nb_k": 80, "cpu_w": 0.20, "mem_ctx": 0.001},
    "Service Request":           {"bh_rate": 24.261, "msgs":  4, "nb_k": 50, "cpu_w": 1.00, "mem_ctx": 0.002},
    "PS Paging":                 {"bh_rate": 10.666, "msgs":  2, "nb_k": 45, "cpu_w": 0.15, "mem_ctx": 0.000},
    "N2 Release":                {"bh_rate": 30.000, "msgs":  2, "nb_k": 55, "cpu_w": 0.89, "mem_ctx": 0.000},
    "Inter-AMF N2 Handover":     {"bh_rate": 1.567,  "msgs": 10, "nb_k": 35, "cpu_w": 0.80, "mem_ctx": 0.005},
    "Intra-AMF N2 Handover":     {"bh_rate": 6.267,  "msgs":  6, "nb_k": 40, "cpu_w": 0.70, "mem_ctx": 0.003},
    "Intra-AMF Xn Handover":     {"bh_rate": 14.099, "msgs":  4, "nb_k": 45, "cpu_w": 0.60, "mem_ctx": 0.002},
    "PDU Session Establishment": {"bh_rate": 1.359,  "msgs":  6, "nb_k": 60, "cpu_w": 0.70, "mem_ctx": 0.001},
    "PDU Session Release":       {"bh_rate": 0.507,  "msgs":  4, "nb_k": 55, "cpu_w": 0.40, "mem_ctx": 0.000},
    "PDU Session Modification":  {"bh_rate": 2.063,  "msgs":  4, "nb_k": 60, "cpu_w": 0.55, "mem_ctx": 0.001},
    "VoNR Voice Call":           {"bh_rate": 0.2293, "msgs":  4, "nb_k": 30, "cpu_w": 0.75, "mem_ctx": 0.003},
    "EPS Fallback Voice":        {"bh_rate": 0.6412, "msgs":  4, "nb_k": 35, "cpu_w": 0.60, "mem_ctx": 0.002},
    "SMS":                       {"bh_rate": 0.4279, "msgs":  4, "nb_k": 40, "cpu_w": 0.25, "mem_ctx": 0.001},
    "5GS-to-EPS Mobility":       {"bh_rate": 1.000,  "msgs":  8, "nb_k": 30, "cpu_w": 0.80, "mem_ctx": 0.004},
    "EPS-to-5GS Mobility":       {"bh_rate": 3.000,  "msgs":  8, "nb_k": 35, "cpu_w": 0.80, "mem_ctx": 0.004},
    "5GS-to-EPS HO (N26)":       {"bh_rate": 2.000,  "msgs": 10, "nb_k": 28, "cpu_w": 0.90, "mem_ctx": 0.005},
    "EPS-to-5GS HO (N26)":       {"bh_rate": 0.000,  "msgs": 10, "nb_k": 28, "cpu_w": 0.90, "mem_ctx": 0.005},
}

# ─── Default slice population mix ──────────────────────────────────────────
SERVICE_MIX = {"eMBB": 0.70, "mMTC": 0.20, "URLLC": 0.10}

# ─── Per-slice arrival rate multipliers ────────────────────────────────────
# Ref: 3GPP TR 22.261 §6, TS 23.501 §5.15 (network slicing behavioural traits)
SLICE_PROC_SCALE: Dict[str, Dict[str, float]] = {
    "eMBB": {
        "Initial Registration": 1.00, "Deregistration": 1.00,
        "Inter-AMF Mobility Reg": 1.00, "Intra-AMF Mobility Reg": 1.00,
        "Periodic Registration": 0.80, "Service Request": 1.20,
        "PS Paging": 1.30, "N2 Release": 1.20,
        "Inter-AMF N2 Handover": 1.00, "Intra-AMF N2 Handover": 1.00,
        "Intra-AMF Xn Handover": 1.00, "PDU Session Establishment": 1.30,
        "PDU Session Release": 1.20, "PDU Session Modification": 1.00,
        "VoNR Voice Call": 1.20, "EPS Fallback Voice": 1.30,
        "SMS": 1.10, "5GS-to-EPS Mobility": 0.80,
        "EPS-to-5GS Mobility": 0.80, "5GS-to-EPS HO (N26)": 0.80,
        "EPS-to-5GS HO (N26)": 0.80,
    },
    "mMTC": {
        "Initial Registration": 0.60, "Deregistration": 0.40,
        "Inter-AMF Mobility Reg": 0.10, "Intra-AMF Mobility Reg": 0.15,
        "Periodic Registration": 2.50, "Service Request": 0.25,
        "PS Paging": 0.10, "N2 Release": 0.30,
        "Inter-AMF N2 Handover": 0.05, "Intra-AMF N2 Handover": 0.08,
        "Intra-AMF Xn Handover": 0.05, "PDU Session Establishment": 0.20,
        "PDU Session Release": 0.20, "PDU Session Modification": 0.10,
        "VoNR Voice Call": 0.00, "EPS Fallback Voice": 0.00,
        "SMS": 0.05, "5GS-to-EPS Mobility": 0.05,
        "EPS-to-5GS Mobility": 0.05, "5GS-to-EPS HO (N26)": 0.02,
        "EPS-to-5GS HO (N26)": 0.02,
    },
    "URLLC": {
        "Initial Registration": 0.80, "Deregistration": 0.60,
        "Inter-AMF Mobility Reg": 1.50, "Intra-AMF Mobility Reg": 1.30,
        "Periodic Registration": 1.20, "Service Request": 0.60,
        "PS Paging": 0.15, "N2 Release": 0.50,
        "Inter-AMF N2 Handover": 2.00, "Intra-AMF N2 Handover": 1.80,
        "Intra-AMF Xn Handover": 2.20, "PDU Session Establishment": 0.70,
        "PDU Session Release": 0.60, "PDU Session Modification": 1.20,
        "VoNR Voice Call": 0.30, "EPS Fallback Voice": 0.10,
        "SMS": 0.05, "5GS-to-EPS Mobility": 1.20,
        "EPS-to-5GS Mobility": 1.20, "5GS-to-EPS HO (N26)": 1.50,
        "EPS-to-5GS HO (N26)": 1.50,
    },
}

# ─── Per-slice CPU weight multipliers ──────────────────────────────────────
# URLLC procedures have stricter preemption-handling and fast-path processing
# that increases instruction counts; mMTC uses a lightweight state machine.
SLICE_CPU_MULT: Dict[str, float] = {
    "eMBB":  1.00,   # reference
    "mMTC":  0.55,   # lightweight IoT state machine, no QoS enforcement
    "URLLC": 1.35,   # fast-path preemption, strict QoS enforcement, crypto overhead
}

# ─── Per-slice UE context memory footprint (MB per active UE) ──────────────
# eMBB:  ~5 KB NAS state + 3 KB PDU ref + 1 KB QoS = ~9 KB  → 0.009 MB
# mMTC:  minimal state, no PDU anchors, ~3 KB               → 0.003 MB
# URLLC: NAS + QoS guarantee tables + pre-alloc buffers ~14 KB → 0.014 MB
SLICE_MEM_PER_UE_MB: Dict[str, float] = {
    "eMBB":  0.009,
    "mMTC":  0.003,
    "URLLC": 0.014,
}

# ─── Per-slice latency baseline (ms) and Log-Normal CV ─────────────────────
# Ref: 3GPP TS 22.261 Table 7.1 (one-way latency requirements)
#   eMBB:   10–100 ms acceptable for control-plane
#   mMTC:   relaxed, 10 s acceptable → low-frequency so no queueing pressure
#   URLLC:  ≤1 ms user-plane; control plane still ~5–10 ms at AMF but tighter
SLICE_LAT_PARAMS: Dict[str, Dict[str, float]] = {
    "eMBB":  {"base_ms": 1.0,  "cv": 0.30, "sla_ms": 100.0},  # 3GPP §6.3.1
    "mMTC":  {"base_ms": 2.0,  "cv": 0.50, "sla_ms": 6000.0}, # relaxed
    "URLLC": {"base_ms": 0.5,  "cv": 0.15, "sla_ms": 5.0},    # strict
}

# ─── Per-slice UE Markov transition probabilities ────────────────────────────
# Calibrated against 3GPP TR 38.913, Shafiq et al. (2012) operator data,
# and Liu et al. IMC 2025 (real 5GC active UE fractions).
#
# Steady-state CM-CONNECTED fraction π_C = p_IC / (p_IC + p_CI)
# p_IC(load) = p_ic_base + p_ic_load × load   (increases with traffic)
# p_CI(load) = p_ci_base + p_ci_load × load   (decreases with traffic, note negative p_ci_load)
#
# Target fractions (3GPP TR 38.913 §7.1 + real operator surveys):
#   eMBB:  ~40-45% CM-Connected at peak hours, ~30% off-peak
#   mMTC:  ~2-3%  CM-Connected (devices report then return to deep sleep)
#   URLLC: ~85-90% CM-Connected (latency-critical: nearly always active)
SLICE_UE_TRANSITIONS: Dict[str, Dict[str, float]] = {
    "eMBB":  {"p_ic_base": 0.0800, "p_ic_load": 0.1600,   # π_C: 32% night → 45% peak
              "p_ci_base": 0.3200, "p_ci_load": -0.1200},
    "mMTC":  {"p_ic_base": 0.0032, "p_ic_load": 0.0088,   # π_C: ~1% night → ~2.5% peak
              "p_ci_base": 0.3968, "p_ci_load": -0.0088}, # IoT: connect briefly then sleep
    "URLLC": {"p_ic_base": 0.3200, "p_ic_load": 0.0800,   # π_C: ~83% night → ~88% peak
              "p_ci_base": 0.0800, "p_ci_load": -0.0400}, # mission-critical: stay connected
}

# ─── Anomaly intensity multiplier tables ───────────────────────────────────
INTENSITY_TABLE: Dict[str, Dict[str, Dict[str, float]]] = {
    "cpu_overload": {
        "mild":     {"cpu": 1.15, "mem": 1.00, "lat": 1.10, "succ": 0.95, "req": 1.00},
        "moderate": {"cpu": 1.35, "mem": 1.05, "lat": 1.25, "succ": 0.85, "req": 1.00},
        "severe":   {"cpu": 1.70, "mem": 1.10, "lat": 1.50, "succ": 0.60, "req": 1.00},
    },
    "memory_leak": {
        "mild":     {"cpu": 1.00, "mem": 1.10, "lat": 1.05, "succ": 0.98, "req": 1.00},
        "moderate": {"cpu": 1.05, "mem": 1.25, "lat": 1.15, "succ": 0.95, "req": 1.00},
        "severe":   {"cpu": 1.10, "mem": 1.45, "lat": 1.30, "succ": 0.85, "req": 1.00},
    },
    "registration_storm": {
        "mild":     {"cpu": 1.15, "mem": 1.00, "lat": 1.10, "succ": 0.90, "req": 1.20},
        "moderate": {"cpu": 1.25, "mem": 1.05, "lat": 1.25, "succ": 0.85, "req": 1.50},
        "severe":   {"cpu": 1.45, "mem": 1.10, "lat": 1.50, "succ": 0.60, "req": 1.80},
    },
    "paging_flood": {
        "mild":     {"cpu": 1.10, "mem": 1.00, "lat": 1.10, "succ": 0.90, "req": 1.25},
        "moderate": {"cpu": 1.18, "mem": 1.00, "lat": 1.22, "succ": 0.80, "req": 1.60},
        "severe":   {"cpu": 1.30, "mem": 1.00, "lat": 1.40, "succ": 0.65, "req": 2.00},
    },
    "handover_failure": {
        "mild":     {"cpu": 1.05, "mem": 1.00, "lat": 1.10, "succ": 0.88, "req": 1.00},
        "moderate": {"cpu": 1.10, "mem": 1.00, "lat": 1.20, "succ": 0.75, "req": 1.00},
        "severe":   {"cpu": 1.20, "mem": 1.00, "lat": 1.45, "succ": 0.50, "req": 1.00},
    },
    "ddos_fake_registrations": {
        "mild":     {"cpu": 1.30, "mem": 1.20, "lat": 1.25, "succ": 0.75, "req": 1.50},
        "moderate": {"cpu": 1.45, "mem": 1.30, "lat": 1.35, "succ": 0.60, "req": 2.00},
        "severe":   {"cpu": 1.75, "mem": 1.40, "lat": 1.60, "succ": 0.40, "req": 3.00},
    },
    "nas_replay_attack": {
        "mild":     {"cpu": 1.10, "mem": 1.00, "lat": 1.10, "succ": 0.90, "req": 1.00},
        "moderate": {"cpu": 1.18, "mem": 1.00, "lat": 1.25, "succ": 0.80, "req": 1.00},
        "severe":   {"cpu": 1.28, "mem": 1.00, "lat": 1.50, "succ": 0.60, "req": 1.00},
    },
    "signaling_storm": {
        "mild":     {"cpu": 1.20, "mem": 1.05, "lat": 1.15, "succ": 0.92, "req": 1.30},
        "moderate": {"cpu": 1.35, "mem": 1.10, "lat": 1.30, "succ": 0.82, "req": 1.60},
        "severe":   {"cpu": 1.60, "mem": 1.20, "lat": 1.55, "succ": 0.65, "req": 2.20},
    },
    # v14 new anomaly types
    "slice_isolation_failure": {
        "mild":     {"cpu": 1.10, "mem": 1.15, "lat": 1.20, "succ": 0.88, "req": 1.10},
        "moderate": {"cpu": 1.20, "mem": 1.25, "lat": 1.40, "succ": 0.75, "req": 1.20},
        "severe":   {"cpu": 1.35, "mem": 1.35, "lat": 1.70, "succ": 0.55, "req": 1.30},
    },
    "amf_overload_cascade": {
        "mild":     {"cpu": 1.25, "mem": 1.10, "lat": 1.30, "succ": 0.85, "req": 1.40},
        "moderate": {"cpu": 1.50, "mem": 1.20, "lat": 1.50, "succ": 0.70, "req": 1.80},
        "severe":   {"cpu": 1.85, "mem": 1.30, "lat": 1.80, "succ": 0.45, "req": 2.50},
    },
}

# ─── Default anomaly injection scenarios ───────────────────────────────────
ANOMALY_SCENARIOS: List[Dict] = [
    # ── Calibrated for 1 AMF instance, 60-day dataset ──────────────────────
    # 24 scenarios × ~2-day spacing → ~210 anomaly slots (~3.6% of 5 760 rows)
    # 3 complete cycles of all 8 anomaly types for balanced class representation
    # All assigned to AMF_00 so single-instance runs capture every type
    # ── Cycle 1 (days 2–16) ────────────────────────────────────────────────
    {"type":"cpu_overload",           "instance":"AMF_00","day": 2,"start":"09:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.25},
    {"type":"memory_leak",            "instance":"AMF_00","day": 4,"start":"14:00","duration_h":3.0,"intensity":"moderate","ramp_h":0.50},
    {"type":"registration_storm",     "instance":"AMF_00","day": 6,"start":"20:00","duration_h":1.0,"intensity":"moderate","ramp_h":0.15},
    {"type":"signaling_storm",        "instance":"AMF_00","day": 8,"start":"03:00","duration_h":1.5,"intensity":"moderate","ramp_h":0.20},
    {"type":"handover_failure",       "instance":"AMF_00","day":10,"start":"11:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.30},
    {"type":"ddos_fake_registrations","instance":"AMF_00","day":12,"start":"17:00","duration_h":1.0,"intensity":"moderate","ramp_h":0.15},
    {"type":"slice_isolation_failure","instance":"AMF_00","day":14,"start":"08:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.25},
    {"type":"amf_overload_cascade",   "instance":"AMF_00","day":16,"start":"22:00","duration_h":1.0,"intensity":"severe",  "ramp_h":0.15},
    # ── Cycle 2 (days 18–32) ───────────────────────────────────────────────
    {"type":"cpu_overload",           "instance":"AMF_00","day":18,"start":"09:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.25},
    {"type":"memory_leak",            "instance":"AMF_00","day":20,"start":"14:00","duration_h":3.0,"intensity":"moderate","ramp_h":0.50},
    {"type":"registration_storm",     "instance":"AMF_00","day":22,"start":"20:00","duration_h":1.0,"intensity":"moderate","ramp_h":0.15},
    {"type":"signaling_storm",        "instance":"AMF_00","day":24,"start":"03:00","duration_h":1.5,"intensity":"moderate","ramp_h":0.20},
    {"type":"handover_failure",       "instance":"AMF_00","day":26,"start":"11:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.30},
    {"type":"ddos_fake_registrations","instance":"AMF_00","day":28,"start":"17:00","duration_h":1.0,"intensity":"moderate","ramp_h":0.15},
    {"type":"slice_isolation_failure","instance":"AMF_00","day":30,"start":"08:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.25},
    {"type":"amf_overload_cascade",   "instance":"AMF_00","day":32,"start":"22:00","duration_h":1.0,"intensity":"severe",  "ramp_h":0.15},
    # ── Cycle 3 (days 34–48) ───────────────────────────────────────────────
    {"type":"cpu_overload",           "instance":"AMF_00","day":34,"start":"09:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.25},
    {"type":"memory_leak",            "instance":"AMF_00","day":36,"start":"14:00","duration_h":3.0,"intensity":"moderate","ramp_h":0.50},
    {"type":"registration_storm",     "instance":"AMF_00","day":38,"start":"20:00","duration_h":1.0,"intensity":"moderate","ramp_h":0.15},
    {"type":"signaling_storm",        "instance":"AMF_00","day":40,"start":"03:00","duration_h":1.5,"intensity":"moderate","ramp_h":0.20},
    {"type":"handover_failure",       "instance":"AMF_00","day":42,"start":"11:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.30},
    {"type":"ddos_fake_registrations","instance":"AMF_00","day":44,"start":"17:00","duration_h":1.0,"intensity":"moderate","ramp_h":0.15},
    {"type":"slice_isolation_failure","instance":"AMF_00","day":46,"start":"08:00","duration_h":2.0,"intensity":"moderate","ramp_h":0.25},
    {"type":"amf_overload_cascade",   "instance":"AMF_00","day":48,"start":"22:00","duration_h":1.0,"intensity":"severe",  "ramp_h":0.15},
]


In [5]:
# ── Benchmark config verification ─────────────────────────────────────
assert DURATION_HOURS == 1440, f'Wrong duration: {DURATION_HOURS} (need 1440 for 60-day benchmark)'
assert AMF_INSTANCES  == 1,    f'Wrong instances: {AMF_INSTANCES} (need 1 for released benchmark)'
assert RANDOM_SEED    == 42,   f'Wrong seed: {RANDOM_SEED}'
expected_rows = (DURATION_HOURS * 60 // STEP_MIN) * AMF_INSTANCES
print(f'✓ Config verified: {DURATION_HOURS//24} days × {AMF_INSTANCES} instance = {expected_rows:,} expected rows')
print(f'  HURST_EXPONENT = {HURST_EXPONENT}  |  NUM_UES = {NUM_UES:,}  |  VCPUS = {VCPUS_PER_AMF}')

✓ Config verified: 60 days × 1 instance = 5,760 expected rows
  HURST_EXPONENT = 0.75  |  NUM_UES = 100,000  |  VCPUS = 8


---
## Section 2 — Statistical Utilities  (NB · Log-Normal · Erlang-C · fGn · GARCH · Sigmoid)

In [6]:
class StatUtils:
    """
    Centralised statistical sampling utilities.

    All methods are stateless classmethods so the caller manages the RNG
    state (enables exact reproducibility with different seeds per AMF instance).
    """

    @staticmethod
    def sample_nb(mean: float, k: float, rng: np.random.RandomState) -> int:
        """
        Negative-Binomial draw: Var = mean + mean²/k.
        At k→∞ converges to Poisson; k=1 is geometric.
        Ref: overdispersed control-plane signalling counts (Charitably 2022).
        """
        mean = max(mean, 0.0)
        if mean < 1e-9:
            return 0
        p = k / (k + mean)
        return int(nbinom.rvs(n=k, p=p, random_state=rng))

    @staticmethod
    def sample_lognormal_ms(mean_ms: float, cv: float,
                             rng: np.random.RandomState) -> float:
        """
        Log-Normal latency sample.
        σ² = ln(1 + cv²),  μ_ln = ln(mean_ms) - σ²/2
        """
        if mean_ms <= 0:
            return 0.0
        sigma2  = math.log(1.0 + cv ** 2)
        mu_ln   = math.log(mean_ms) - sigma2 / 2.0
        return float(rng.lognormal(mu_ln, math.sqrt(sigma2)))

    @staticmethod
    def erlang_c(c: int, rho: float) -> float:
        """
        Erlang-C blocking probability P(wait > 0) for M/M/c queue (rho < 1).
        Uses the standard numerically stable recursion.
        Ref: Kleinrock (1975) Queueing Systems Vol. 1.
        """
        rho = min(max(rho, 1e-9), 0.999_999)
        a   = c * rho           # offered traffic in Erlangs
        # Compute Σ_{k=0}^{c-1} a^k / k!  iteratively
        sum_term = 0.0
        term     = 1.0
        for k in range(1, c):
            term *= a / k
            sum_term += term
        sum_term += 1.0          # k=0 term
        top  = (a ** c) / math.factorial(c) / (1.0 - rho)
        return top / (sum_term + top) if (sum_term + top) > 0 else 1.0

    @staticmethod
    def mmc_sojourn(lam: float, mu: float, c: int) -> Tuple[float, float, float]:
        """
        M/M/c mean sojourn time T_s (seconds) and mean waiting time W_q.
        Returns (T_s, W_q, rho).

        T_s = W_q + 1/μ,  W_q = C(c,ρ) / (c·μ − λ)
        where C(c,ρ) is the Erlang-C value.
        """
        lam = max(lam, 1e-9)
        rho = min(max(lam / (c * mu), 1e-9), 0.999_999)
        C   = StatUtils.erlang_c(c, rho)
        Wq  = C / max(c * mu - lam, 1e-9)
        Ts  = Wq + 1.0 / mu
        return Ts, Wq, rho

    @staticmethod
    def jackson_throughput(lam_classes: Dict[str, float],
                            mu: float, c: int) -> Dict[str, float]:
        """
        Approximate per-class throughput in a Jackson open network node.
        Uses BCMP theorem: with class-independent service rates the
        aggregate arrival Λ = Σ λ_i, and each class sees fraction
        λ_i / Λ of the server capacity.
        Ref: Baskett, Chandy, Muntz, Palacios (1975).
        Returns per-class effective throughput (events/s, capped at μ·c).
        """
        lam_total = sum(lam_classes.values())
        if lam_total < 1e-9:
            return {k: 0.0 for k in lam_classes}
        rho   = lam_total / max(c * mu, 1e-9)
        gamma = min(1.0, 1.0 / max(rho, 1e-9))   # throughput ratio
        return {k: v * gamma for k, v in lam_classes.items()}

    @staticmethod
    def fractional_gaussian_noise(n: int, H: float,
                                   rng: np.random.RandomState) -> np.ndarray:
        """
        Approximate fGn via spectral / Davies-Harte method.
        H ∈ (0.5, 1) → long-range dependence (self-similar traffic).
        Ref: Norros (1994) "A storage model with self-similar input".
        """
        f      = np.fft.rfftfreq(n)
        f[0]   = 1.0
        psd    = f ** (-(2 * H - 1))
        psd[0] = 0.0
        phases  = rng.uniform(0.0, 2.0 * np.pi, len(psd))
        spec    = np.sqrt(psd) * np.exp(1j * phases)
        noise   = np.fft.irfft(spec, n=n)
        return (noise - noise.mean()) / (noise.std() + 1e-9)

    @staticmethod
    def garch_volatility(n: int, omega: float, alpha: float, beta: float,
                          rng: np.random.RandomState) -> np.ndarray:
        """
        GARCH(1,1) conditional standard deviation sequence.
        h_t = ω + α·ε²_{t-1}·h_{t-1} + β·h_{t-1}
        Ref: Papagiannaki et al. (2003) – IP traffic volatility clustering.
        """
        h       = np.zeros(n)
        eps     = rng.standard_normal(n)
        h[0]    = omega / max(1.0 - alpha - beta, 1e-9)
        for t in range(1, n):
            h[t] = omega + alpha * (eps[t - 1] ** 2) * h[t - 1] + beta * h[t - 1]
        return np.sqrt(np.clip(h, 1e-9, None))

    @staticmethod
    def sigmoid_ramp(elapsed_h: float, total_h: float, ramp_h: float) -> float:
        """
        v14 NEW: Sigmoid onset / recovery envelope for anomaly intensity.
        Returns a value in [0, 1].
        - Rises from 0 → 1 over the first ramp_h hours (logistic onset)
        - Flat at 1 during the middle phase
        - Falls from 1 → 0 over the last ramp_h hours (logistic recovery)

        σ(x) = 1 / (1 + exp(-k·x))  with k chosen so 0.99 is reached at ramp_h/2.
        """
        if total_h <= 0:
            return 0.0
        k = 9.0 / max(ramp_h, 1e-3)    # k: steepness of logistic curve

        # onset ramp in [0, ramp_h)
        onset  = 1.0 / (1.0 + math.exp(-k * (elapsed_h - ramp_h / 2.0)))
        # recovery ramp: mirror in [total_h - ramp_h, total_h)
        remaining_h = total_h - elapsed_h
        recovery = 1.0 / (1.0 + math.exp(-k * (remaining_h - ramp_h / 2.0)))

        return float(np.clip(min(onset, recovery), 0.0, 1.0))

# ---

# ──────────────────────────────────────────────────────────────────────
# Section 3 — Temporal Load Engine  (Diurnal · DoW · fGn · GARCH)
# ──────────────────────────────────────────────────────────────────────

---
## Section 3 — Temporal Load Engine  (Diurnal · DoW · fGn · GARCH)

In [7]:
class TemporalEngine:
    """
    Produces normalised traffic load λ̃(t) ∈ [0, 1].
    Combines class-specific diurnal shape, day-of-week modulation,
    fractional Gaussian noise (long-range dependence), and GARCH
    volatility clustering (busy-hour bursts).

    The final load curve is:
        λ̃(t) = clip[ D(t,cls) · DoW(t) · (1 + 0.15·fGn(t)) · (0.85 + 0.15·GARCH(t)) ]

    Ref: Shafiq et al. (2012), Xu et al. (2011), Botta et al. (2016).
    """

    def __init__(self, seed: int, H: float = HURST_EXPONENT):
        self.rng = np.random.RandomState(seed)
        self.H   = H

    @staticmethod
    def _gaussian_peak(h: np.ndarray, center: float,
                        sigma: float, height: float) -> np.ndarray:
        return height * np.exp(-((h - center) ** 2) / (2.0 * sigma ** 2))

    def diurnal_shape(self, hours: np.ndarray, cls: str = "eMBB",
                       is_weekend: bool = False) -> np.ndarray:
        """
        Class-specific dual-peak diurnal, normalised to [0, 1].

        Weekday shapes:
          eMBB:  broad business-hours plateau with evening shoulder.
                 Calibrated jointly against:
                   - Telecom Italia CDR (Barlacchi et al. 2015): r=0.983, MAPE=5.7%
                   - 5G3E real AMF CPU  (Phung et al. 2022):     r=0.991, MAPE=4.5%
                 σ widened 1.3→2.6 (morning) to produce sustained h08-h18 plateau
                 matching real network sustained load (Shafiq 2012 Fig. 4).
          mMTC:  near-flat with IoT micro-burst every ~4 hours
          URLLC: industrial daytime ramps at 10:00 and 15:00

        Weekend shapes (is_weekend=True):
          eMBB:  later wake-up, single broad midday peak, strong evening leisure peak.
                 Ref: Shafiq et al. (2012) Fig. 6 weekend vs weekday comparison.
          mMTC:  slightly reduced (fewer automated industrial triggers on weekends)
          URLLC: minimal — factories offline; only residual healthcare/transport load
        """
        base = 0.10 + 0.03 * np.sin(2 * np.pi * hours / 24.0)

        if not is_weekend:
            # ── Weekday shapes ────────────────────────────────────────────────
            if cls == "eMBB":
                base += self._gaussian_peak(hours,  9.5, 2.6, 0.48)  # broad morning plateau
                base += self._gaussian_peak(hours, 17.0, 3.7, 0.60)  # evening shoulder
            elif cls == "mMTC":
                base  = 0.55 + 0.15 * np.sin(2 * np.pi * hours / 24.0)
                micro = 0.08 * (1 + 0.5 * np.sin(2 * np.pi * hours))  # ~1h IoT microburst
                base += micro
            elif cls == "URLLC":
                base += self._gaussian_peak(hours, 10.0, 3.0, 0.35)   # factory AM ramp
                base += self._gaussian_peak(hours, 15.0, 2.5, 0.25)   # factory PM ramp
        else:
            # ── Weekend shapes ────────────────────────────────────────────────
            # Refs: Shafiq et al. (2012), Barlacchi et al. (2015) weekend CDR profiles
            if cls == "eMBB":
                # Later wake-up, single broad midday peak, strong evening leisure peak
                base += self._gaussian_peak(hours, 12.0, 3.5, 0.50)  # lazy morning/noon
                base += self._gaussian_peak(hours, 20.0, 3.0, 0.65)  # prime-time streaming
                base += self._gaussian_peak(hours,  0.5, 1.5, 0.15)  # late-night social
            elif cls == "mMTC":
                # IoT devices mostly follow fixed schedules — modest weekend reduction
                base  = 0.48 + 0.12 * np.sin(2 * np.pi * hours / 24.0)
                micro = 0.06 * (1 + 0.5 * np.sin(2 * np.pi * hours))  # reduced bursts
                base += micro
            elif cls == "URLLC":
                # Factories offline — only residual healthcare/transport/utilities load
                base += self._gaussian_peak(hours, 10.0, 4.0, 0.15)  # reduced residual
                base += self._gaussian_peak(hours, 18.0, 3.0, 0.12)  # evening transport

        base = np.clip(base, 0.05, None)
        return base / base.max()

    @staticmethod
    def dow_factor(dow: int, proc: str) -> float:
        """
        Day-of-week multiplier.  Mon=0 … Sun=6.
        Weekends: fewer commuter handovers, paging still elevated (leisure).
        Ref: Botta et al. (2016) weekly cycles.
        """
        if dow in (5, 6):
            return {"handover": 0.65, "paging": 0.88, "reg": 0.80,
                    "pdu": 0.85}.get(proc, 0.85)
        return {"handover": 1.10}.get(proc, 1.00)

    def build_load_curve(self, n: int, start_dt: datetime,
                          cls: str = "eMBB", proc: str = "reg") -> np.ndarray:
        """Build the full n-slot normalised load curve λ̃(t).

        Implements: λ̃_s(t) = D_s(t) · DoW(t) · (1 + 0.15·fGn(t)) · (0.85 + 0.15·vol(t))
        where D_s(t) is slice-specific (weekday or weekend shape per slot) and
        DoW(t) is a 7-day procedure-type multiplier calibrated from mobile CDR studies.
        Ref: Shafiq et al. (2012), Barlacchi et al. (2015), Botta et al. (2016).
        """
        _sm = getattr(self, 'step_min', STEP_MIN)  # respect runtime step_min override
        slot_h = np.array([
            (start_dt + timedelta(minutes=i * _sm + _sm / 2)).hour
            + (start_dt + timedelta(minutes=i * _sm + _sm / 2)).minute / 60.0
            for i in range(n)
        ]) % 24.0
        dows = np.array([
            (start_dt + timedelta(minutes=i * _sm)).weekday()
            for i in range(n)
        ])
        # Per-slot diurnal shape — weekday (Mon–Fri) and weekend (Sat–Sun) differ
        is_weekend_arr = np.array([(d in (5, 6)) for d in dows])
        # Pre-compute full 24h shapes for both day types (vectorised, one call each)
        _h24         = np.arange(24, dtype=float)
        _shape_wkday = self.diurnal_shape(_h24, cls, is_weekend=False)
        _shape_wkend = self.diurnal_shape(_h24, cls, is_weekend=True)
        # Map each slot's fractional hour to the correct shape with linear interpolation
        _hidx      = np.floor(slot_h).astype(int) % 24
        _hidx_next = (_hidx + 1) % 24
        _frac      = slot_h - np.floor(slot_h)
        base = np.where(is_weekend_arr,
                        _shape_wkend[_hidx] * (1 - _frac) + _shape_wkend[_hidx_next] * _frac,
                        _shape_wkday[_hidx] * (1 - _frac) + _shape_wkday[_hidx_next] * _frac)
        dow_mult = np.array([self.dow_factor(d, proc) for d in dows])
        base    *= dow_mult
        fgn      = StatUtils.fractional_gaussian_noise(n, self.H, self.rng)
        base    *= (1.0 + 0.15 * fgn)
        garch    = StatUtils.garch_volatility(n, 0.05, 0.15, 0.80, self.rng)
        garch   /= garch.mean()
        base    *= (0.85 + 0.15 * garch)
        return np.clip(base, 0.05, 1.0)

# ---

# ──────────────────────────────────────────────────────────────────────
# Section 4 — Markov Chain UE State Model
# ──────────────────────────────────────────────────────────────────────

---
## Section 4 — Markov Chain UE State Model

In [8]:
class UEStateModel:
    """
    3-state Markov chain for UE population dynamics.
    States: IDLE (0) | CM-CONNECTED (1) | RM-DEREGISTERED (2)

    Transition probabilities are load-dependent AND slice-specific,
    calibrated to match real operator CM-CONNECTED fractions:
      eMBB:  ~40-45% connected at peak  (3GPP TR 38.913)
      mMTC:  ~2-3%   connected          (IoT deep-sleep duty cycle)
      URLLC: ~85-90% connected          (mission-critical always-on)

    Refs: 3GPP TS 23.501 §5.3; TR 38.913 §7.1;
          Shafiq et al. (2012); Liu et al. IMC 2025.
    """

    def __init__(self, num_ues: int, rng: np.random.RandomState,
                 slice_cls: str = "eMBB"):
        self.num_ues   = num_ues
        self.rng       = rng
        self.slice_cls = slice_cls
        self._params   = SLICE_UE_TRANSITIONS.get(slice_cls,
                             SLICE_UE_TRANSITIONS["eMBB"])
        # Initialise near steady state at moderate load (load=0.5)
        p_ic0  = float(np.clip(self._params["p_ic_base"] +
                               0.5 * self._params["p_ic_load"], 0, 0.99))
        p_ci0  = float(np.clip(self._params["p_ci_base"] +
                               0.5 * self._params["p_ci_load"], 0.001, 0.99))
        ss0    = p_ic0 / (p_ic0 + p_ci0)          # steady-state connected fraction
        n_conn = max(1, int(num_ues * ss0))
        n_dereg = max(0, int(num_ues * 0.03))
        self.idle         = max(0, num_ues - n_conn - n_dereg)
        self.connected    = n_conn
        self.deregistered = n_dereg

    def step(self, load: float) -> Tuple[int, int, int]:
        """
        One-slot Markov transition.  Returns (idle, connected, deregistered).
        Transition probabilities are slice-specific and load-dependent:
          p_IC(load) = p_ic_base + p_ic_load × load
          p_CI(load) = p_ci_base + p_ci_load × load  (p_ci_load is negative)
        """
        p = self._params
        p_i2c = float(np.clip(p["p_ic_base"] + p["p_ic_load"] * load, 0.0, 0.99))
        p_c2i = float(np.clip(p["p_ci_base"] + p["p_ci_load"] * load, 0.001, 0.99))
        p_c2d = 0.001   # deregistration: ~0.1% per slot (reduced from 0.2%)
        i2c   = int(self.rng.binomial(self.idle, p_i2c))
        c2i   = int(self.rng.binomial(self.connected, p_c2i))
        c2d   = int(self.rng.binomial(max(self.connected - c2i, 0), p_c2d))
        d2i   = int(self.rng.binomial(self.deregistered, 0.30))
        self.idle         = max(0, self.idle - i2c + c2i + d2i)
        self.connected    = max(0, self.connected + i2c - c2i - c2d)
        self.deregistered = max(0, self.deregistered + c2d - d2i)
        return self.idle, self.connected, self.deregistered

# ---

# ──────────────────────────────────────────────────────────────────────
# Section 5 — Anomaly Engine  (Sigmoid Onset / Recovery)
# ──────────────────────────────────────────────────────────────────────

---
## Section 5 — Anomaly Engine  (Sigmoid Onset / Recovery)

In [9]:
def get_intensity_factors(anom_type: str,
                           intensity: Union[str, Dict[str, float]],
                           sigmoid_weight: float = 1.0) -> Dict[str, float]:
    """
    Return {cpu, mem, lat, succ, req} multipliers scaled by sigmoid_weight ∈ [0,1].
    When sigmoid_weight = 0 → no anomaly; = 1 → full anomaly.
    Intermediate values smoothly interpolate between normal and peak anomaly.
    """
    if isinstance(intensity, dict):
        base = {"cpu": 1.0, "mem": 1.0, "lat": 1.0, "succ": 1.0, "req": 1.0}
        base.update(intensity)
    else:
        table = INTENSITY_TABLE.get(anom_type, {})
        base  = table.get(intensity, {"cpu":1.0,"mem":1.0,"lat":1.0,"succ":1.0,"req":1.0})

    # Blend between identity (w=0) and peak (w=1) using sigmoid_weight
    w = float(np.clip(sigmoid_weight, 0.0, 1.0))
    return {
        "cpu":  1.0 + (base["cpu"]  - 1.0) * w,
        "mem":  1.0 + (base["mem"]  - 1.0) * w,
        "lat":  1.0 + (base["lat"]  - 1.0) * w,
        "succ": 1.0 - (1.0 - base["succ"]) * w,
        "req":  1.0 + (base["req"]  - 1.0) * w,
    }


def build_anomaly_mask(timestamps: List[datetime],
                        instance: str,
                        start_dt: datetime,
                        scenarios: Optional[List[Dict]] = None
                       ) -> List[Optional[Dict]]:
    """
    Build per-slot anomaly mask with sigmoid intensity weights.
    Returns a list of (scenario_dict | None) per timestamp slot.
    Each non-None entry also includes 'sigmoid_weight' key.
    """
    if scenarios is None:
        scenarios = ANOMALY_SCENARIOS
    mask     = [None] * len(timestamps)
    base_day = start_dt.replace(hour=0, minute=0, second=0, microsecond=0)

    for sc in scenarios:
        if sc.get("instance", "all") not in (instance, "all"):
            continue
        hh, mm    = map(int, sc["start"].split(":"))
        day_off    = sc.get("day", 0)
        t0 = base_day + timedelta(days=day_off, hours=hh, minutes=mm)
        t1 = t0 + timedelta(hours=sc.get("duration_h", 1.0))
        ramp_h     = sc.get("ramp_h", 0.25)
        dur_h      = sc.get("duration_h", 1.0)

        for i, ts in enumerate(timestamps):
            if t0 <= ts < t1:
                elapsed_h   = (ts - t0).total_seconds() / 3600.0
                sig_w        = StatUtils.sigmoid_ramp(elapsed_h, dur_h, ramp_h)
                entry        = dict(sc)
                entry["sigmoid_weight"] = sig_w
                mask[i]     = entry
    return mask


---
## Section 6 — Main Dataset Generator  — `AMFDatasetGenerator`

In [10]:
class AMFDatasetGenerator:
    """
    Generates a multi-instance, multi-slice synthetic AMF KPI dataset.

    Architecture:
      TemporalEngine   – per-slice normalised load curve λ̃(t)
      UEStateModel     – per-instance Markov UE population
      StatUtils        – NB / LogNormal / Erlang-C / fGn / GARCH / Sigmoid
      AnomalyEngine    – sigmoid-ramped fault scenarios
      Per-slice CPU/Memory/Latency computation (v14 key improvement)
    """

    # Reference service rate: 1 vCPU handles ~120 "equiv-Service-Requests" / s
    _MU_SRV   = 120.0                  # events/s per vCPU (Service Request equiv.)
    _SLOT_S   = STEP_MIN * 60.0        # seconds per observation slot

    def __init__(
        self,
        seed:              int   = RANDOM_SEED,
        amf_instances:     int   = AMF_INSTANCES,
        ue_embb:           int   = 70_000,
        ue_mmtc:           int   = 20_000,
        ue_urllc:          int   = 10_000,
        include_anomalies: bool  = True,
        anomaly_scenarios: Optional[List[Dict]] = None,
        vcpus_per_amf:     int   = VCPUS_PER_AMF,   # vCPUs per AMF instance
        mem_max_mb:        float = MEM_MAX_MB,        # RAM ceiling per AMF instance (MB)
        duration_hours:    int   = DURATION_HOURS,    # dataset length in hours
        step_min:          int   = STEP_MIN,           # observation slot in minutes
    ):
        """
        Parameters
        ----------
        vcpus_per_amf : int
            Number of virtual CPUs allocated to each AMF VNF instance.
            Controls the service capacity (μ) of the M/M/c queue and therefore
            the CPU utilisation level at a given load.
            Typical values:
              4   — small lab / Open5GS Docker (IEEE 10885600 testbed)
              8   — cloud-native mid-tier pod  (default)
              16  — larger Kubernetes node
              32+ — carrier-grade bare-metal
        mem_max_mb : float
            RAM ceiling per AMF instance in MB.  Memory utilisation is reported
            as a percentage of this value.
            Typical values:
              4096  — 4 GB  (small lab)
              8192  — 8 GB  (default)
              16384 — 16 GB (mid-tier cloud)
              65536 — 64 GB (carrier-grade)
        """
        self.seed              = seed
        self.amf_instances     = max(1, int(amf_instances))
        self.ue_embb           = max(0, int(ue_embb))
        self.ue_mmtc           = max(0, int(ue_mmtc))
        self.ue_urllc          = max(0, int(ue_urllc))
        self.num_ues           = self.ue_embb + self.ue_mmtc + self.ue_urllc
        self.include_anomalies = bool(include_anomalies)
        self.vcpus_per_amf     = max(1, int(vcpus_per_amf))
        self.mem_max_mb        = float(mem_max_mb)
        self.duration_hours    = max(24, int(duration_hours))
        self.step_min          = int(step_min)
        if self.step_min not in (1, 5, 15, 30, 60):
            raise ValueError(f'step_min must be 1/5/15/30/60, got {self.step_min}')
        if anomaly_scenarios is not None:
            self.anomaly_scenarios = anomaly_scenarios
        elif include_anomalies:
            self.anomaly_scenarios = ANOMALY_SCENARIOS
        else:
            self.anomaly_scenarios = []

        total = max(self.num_ues, 1)
        self.service_mix = {
            "eMBB":  self.ue_embb  / total,
            "mMTC":  self.ue_mmtc  / total,
            "URLLC": self.ue_urllc / total,
        }
        self.rng = np.random.RandomState(seed)
        self.te  = TemporalEngine(seed=seed + 1, H=HURST_EXPONENT)

    # ── helpers ──────────────────────────────────────────────────────────

    def _slice_mean(self, event_key: str, cls: str,
                     n_ues_cls: int, n_amf: int, load: float) -> float:
        """
        λ(event, cls, t) = bh_rate(event) × SLICE_PROC_SCALE[cls][event]
                         × n_ues_cls/n_amf × (STEP_MIN/60) × load(cls, t)
        """
        ev    = EVENT_DATA[event_key]
        scale = SLICE_PROC_SCALE.get(cls, {}).get(event_key, 1.0)
        mean  = ev["bh_rate"] * scale * n_ues_cls * self.step_min / 60.0 / n_amf
        return mean * load

    def _draw_event_counts(self, event_key: str,
                            load_per_cls: Dict[str, float],
                            n_ues_per_cls: Dict[str, int],
                            n_amf: int,
                            anom_req_mult: float = 1.0) -> int:
        """Σ_cls NB(λ_cls) across slice classes."""
        ev    = EVENT_DATA[event_key]
        total = 0
        for cls, n_ues in n_ues_per_cls.items():
            if n_ues <= 0:
                continue
            load = load_per_cls.get(cls, 0.0)
            mean = self._slice_mean(event_key, cls, n_ues, n_amf, load)
            mean *= anom_req_mult
            total += StatUtils.sample_nb(mean, ev["nb_k"], self.rng)
        return total

    def _noisy_succ(self, base: float) -> float:
        return float(np.clip(self.rng.normal(base, 0.002), 0.01, 1.0))

    # ── per-slice CPU compute ─────────────────────────────────────────────
    def _compute_slice_cpu(self, event_counts: Dict[str, int],
                            vcpus: int, cap_factor: float,
                            afact: Dict, cls: str) -> float:
        """
        CPU utilisation fraction for one slice.
        ρ_cpu(cls) = Σ_proc [count(proc,cls) × cpu_w(proc) × SLICE_CPU_MULT(cls)]
                     / vcpu_capacity_per_slot
        """
        cpu_equiv = 0.0
        cpu_mult  = SLICE_CPU_MULT[cls]
        for ev_key, cnt in event_counts.items():
            w = EVENT_DATA[ev_key]["cpu_w"] * cpu_mult
            cpu_equiv += cnt * w
        _slot_s = getattr(self, 'step_min', STEP_MIN) * 60.0
        lam_equiv  = cpu_equiv / _slot_s
        vcpu_cap   = self._MU_SRV * vcpus * cap_factor
        rho        = lam_equiv / max(vcpu_cap, 1e-9)
        cpu_base   = min(1.0, rho) * 100.0 * afact["cpu"]
        return float(np.clip(cpu_base + self.rng.normal(0.0, 1.2), 0.5, 99.5))

    # ── per-slice memory compute ──────────────────────────────────────────
    def _compute_slice_mem(self, active_ues_cls: int,
                            auth_att_cls: int,
                            queue_len_raw: float,
                            afact: Dict, cls: str) -> float:
        """
        Memory footprint for one slice (MB).
        mem(cls) = mem_base_cls + active_ues_cls × SLICE_MEM_PER_UE_MB[cls]
                 + 0.08 × auth_att_cls + 0.02 × queue_len_raw
        mem_base split proportionally (512 MB / 3 slices weighted by UE fraction).
        """
        mem_per_ue  = SLICE_MEM_PER_UE_MB[cls]
        mem_ctx     = active_ues_cls * mem_per_ue
        mem_auth    = min(60.0, 0.08 * auth_att_cls)
        mem_queue   = 0.02 * queue_len_raw
        mem_total   = mem_ctx + mem_auth + mem_queue
        return float(max(0.0, mem_total * afact["mem"]))

    # ── per-slice NAS latency ─────────────────────────────────────────────
    def _compute_slice_lat(self, Wq_s: float, afact: Dict, cls: str,
                            rng: np.random.RandomState) -> float:
        """
        Per-slice mean control-plane latency.
        T_total(cls) = T_queue + T_proc(cls)
        T_proc(cls) sampled from LogNormal with class-specific base and CV.
        Ref: SLICE_LAT_PARAMS and TS 22.261 Table 7.1.
        """
        params  = SLICE_LAT_PARAMS[cls]
        T_queue = Wq_s * 1000.0 * afact["lat"]   # ms
        T_proc  = StatUtils.sample_lognormal_ms(
            params["base_ms"] * afact["lat"], params["cv"], rng
        )
        return float(T_queue + T_proc)

    # ── slice load diversity (entropy) ────────────────────────────────────
    @staticmethod
    def _slice_entropy(load_cls: Dict[str, float]) -> float:
        """
        Shannon entropy H = -Σ p·log2(p) of normalised per-slice load.
        High entropy → balanced load across slices; low → one slice dominates.
        """
        vals = np.array([max(v, 1e-9) for v in load_cls.values()])
        probs = vals / vals.sum()
        return float(-np.sum(probs * np.log2(probs)))

    # ── main generate ─────────────────────────────────────────────────────
    def generate(self, progress_callback=None) -> pd.DataFrame:
        """
        Main generation loop.
        Returns a tidy DataFrame, one row per (timestamp, amf_instance).

        Parameters
        ----------
        progress_callback : callable(float) | None
            Optional function called with a float in [0, 100] after each AMF
            instance is fully generated.  Useful for Colab progress bars.
        """
        start_dt   = datetime(2024, 1, 1, 0, 0, 0)
        _SLOT_S_eff = self.step_min * 60.0
        n_slots    = (self.duration_hours * 60) // self.step_min
        timestamps = [start_dt + timedelta(minutes=i * self.step_min) for i in range(n_slots)]
        _amf_n     = self.amf_instances
        _num_ues   = self.num_ues
        _svc_mix   = self.service_mix

        # Pre-compute per-class load curves (shared across AMF instances)
        load_curves: Dict[str, np.ndarray] = {
            cls: self.te.build_load_curve(n_slots, start_dt, cls=cls, proc="reg")
            for cls in _svc_mix
        }

        all_rows: List[Dict] = []

        for amf_idx in range(_amf_n):
            inst_id   = f"AMF_{amf_idx:02d}"
            amf_rng   = np.random.RandomState(self.seed + amf_idx * 1000)
            inst_cap  = amf_rng.uniform(0.82, 1.22)   # per-instance capacity jitter
            # ±20% calibrated from real Huawei vUSN AMF operator traces
            # (ATT/SUB diurnal across 3 instances, Aug 2025 — ~±20% inter-instance spread)
            vcpus     = self.vcpus_per_amf
            _ues_cls  = {
                "eMBB":  self.ue_embb  // max(1, _amf_n),
                "mMTC":  self.ue_mmtc  // max(1, _amf_n),
                "URLLC": self.ue_urllc // max(1, _amf_n),
            }
            # One per-slice UE model — each slice has its own transition probabilities
            _ue_models: Dict[str, "UEStateModel"] = {
                cls: UEStateModel(
                    max(1, _ues_cls[cls]),
                    np.random.RandomState(self.seed + amf_idx * 1000 + i),
                    slice_cls=cls,
                )
                for i, cls in enumerate(_svc_mix)
            }
            anom_mask = build_anomaly_mask(
                timestamps, inst_id, start_dt, scenarios=self.anomaly_scenarios
            )

            for t_idx, ts in enumerate(timestamps):
                sc       = anom_mask[t_idx]
                is_anom  = sc is not None
                anom_type = sc["type"]      if is_anom else "none"
                sig_w     = sc.get("sigmoid_weight", 1.0) if is_anom else 0.0
                afact     = get_intensity_factors(
                    anom_type, sc.get("intensity", "moderate"), sig_w
                ) if is_anom else {"cpu":1.0,"mem":1.0,"lat":1.0,"succ":1.0,"req":1.0}

                # Normalised per-class load at this slot
                _load_cls = {
                    cls: float(load_curves[cls][t_idx]) * inst_cap
                    for cls in _svc_mix
                }
                load      = sum(_svc_mix[cls] * _load_cls[cls] for cls in _svc_mix)
                anom_req  = afact["req"]

                # UE state evolution
                # Step each per-slice UE model with its own class-specific load
                _slice_states = {
                    cls: _ue_models[cls].step(_load_cls.get(cls, load))
                    for cls in _svc_mix
                }
                # Aggregate: connected = sum of connected UEs across all slices
                idle         = sum(s[0] for s in _slice_states.values())
                connected    = sum(s[1] for s in _slice_states.values())
                deregistered = sum(s[2] for s in _slice_states.values())
                active_ues   = connected

                # Per-slice connected UE counts (directly from per-slice models)
                ues_conn_cls = {
                    cls: _slice_states[cls][1]   # index 1 = connected
                    for cls in _svc_mix
                }

                # ══ 5.2.1  REGISTRATION MANAGEMENT (RM) ═════════════════
                init_reg_att      = self._draw_event_counts("Initial Registration",      _load_cls, _ues_cls, _amf_n, anom_req)
                inter_mob_reg_att = self._draw_event_counts("Inter-AMF Mobility Reg",    _load_cls, _ues_cls, _amf_n, anom_req)
                intra_mob_reg_att = self._draw_event_counts("Intra-AMF Mobility Reg",    _load_cls, _ues_cls, _amf_n, anom_req)
                per_reg_att       = self._draw_event_counts("Periodic Registration",     _load_cls, _ues_cls, _amf_n, 1.0)
                dereg_att         = self._draw_event_counts("Deregistration",            _load_cls, _ues_cls, _amf_n, 1.0)

                mob_reg_att   = inter_mob_reg_att + intra_mob_reg_att
                total_reg_att = init_reg_att + mob_reg_att + per_reg_att

                sr_init  = self._noisy_succ(0.9985 * afact["succ"])
                sr_mob   = self._noisy_succ(0.9970 * afact["succ"])
                sr_per   = self._noisy_succ(0.9990 * afact["succ"])
                sr_dereg = self._noisy_succ(0.9995 * afact["succ"])

                init_reg_succ  = int(init_reg_att  * sr_init)
                mob_reg_succ   = int(mob_reg_att   * sr_mob)
                per_reg_succ   = int(per_reg_att   * sr_per)
                dereg_succ     = int(dereg_att     * sr_dereg)
                total_reg_succ = init_reg_succ + mob_reg_succ + per_reg_succ

                # ══ 5.2.2  CONNECTION MANAGEMENT (CM) ════════════════════
                srv_req_att  = self._draw_event_counts("Service Request", _load_cls, _ues_cls, _amf_n, anom_req)
                n2_rel_att   = self._draw_event_counts("N2 Release",      _load_cls, _ues_cls, _amf_n, 1.0)
                srv_req_succ = int(srv_req_att * self._noisy_succ(0.9980 * afact["succ"]))
                n2_rel_succ  = int(n2_rel_att  * self._noisy_succ(0.9999 * afact["succ"]))

                # ══ 5.2.3  MOBILITY MANAGEMENT (MM) ══════════════════════
                inter_n2_ho_att = self._draw_event_counts("Inter-AMF N2 Handover",  _load_cls, _ues_cls, _amf_n, anom_req)
                intra_n2_ho_att = self._draw_event_counts("Intra-AMF N2 Handover",  _load_cls, _ues_cls, _amf_n, anom_req)
                intra_xn_ho_att = self._draw_event_counts("Intra-AMF Xn Handover",  _load_cls, _ues_cls, _amf_n, anom_req)
                eps5g_mob       = self._draw_event_counts("EPS-to-5GS Mobility",    _load_cls, _ues_cls, _amf_n, 1.0)
                five_eps_mob    = self._draw_event_counts("5GS-to-EPS Mobility",    _load_cls, _ues_cls, _amf_n, 1.0)
                inter_n26_ho    = self._draw_event_counts("5GS-to-EPS HO (N26)",    _load_cls, _ues_cls, _amf_n, 1.0)
                total_ho_att    = inter_n2_ho_att + intra_n2_ho_att + intra_xn_ho_att

                intra_xn_ho_succ = int(intra_xn_ho_att * self._noisy_succ(0.9940 * afact["succ"]))
                intra_n2_ho_succ = int(intra_n2_ho_att * self._noisy_succ(0.9880 * afact["succ"]))
                inter_n2_ho_succ = int(inter_n2_ho_att * self._noisy_succ(0.9850 * afact["succ"]))
                total_ho_succ    = intra_xn_ho_succ + intra_n2_ho_succ + inter_n2_ho_succ

                # ══ 5.2.4  PAGING (PAG) ═══════════════════════════════════
                paging_att  = self._draw_event_counts("PS Paging", _load_cls, _ues_cls, _amf_n, anom_req)
                sr_pag      = self._noisy_succ(0.9950 * afact["succ"])
                paging_succ = int(paging_att * sr_pag)
                paging_disc = int(paging_att * amf_rng.uniform(0.001, 0.008))
                paging_retry = max(0, int((paging_att - paging_succ - paging_disc)
                                    * amf_rng.uniform(0.5, 1.5)))

                # ══ 5.2.5  UE CONTEXT (UC) ════════════════════════════════
                ctx_created  = init_reg_att + mob_reg_att
                ctx_released = dereg_att + n2_rel_att
                ctx_modified = intra_mob_reg_att + per_reg_att
                active_ctx   = max(0, int(connected * amf_rng.uniform(0.97, 1.03)))
                max_ctx_cap  = int(max(1, _num_ues // _amf_n) * 0.60)

                # ══ 5.2.6  PDU SESSION (SM) ═══════════════════════════════
                pdu_estab_att = self._draw_event_counts("PDU Session Establishment", _load_cls, _ues_cls, _amf_n, anom_req)
                pdu_rel_att   = self._draw_event_counts("PDU Session Release",       _load_cls, _ues_cls, _amf_n, 1.0)
                pdu_mod_att   = self._draw_event_counts("PDU Session Modification",  _load_cls, _ues_cls, _amf_n, 1.0)
                vonr_att      = self._draw_event_counts("VoNR Voice Call",            _load_cls, _ues_cls, _amf_n, 1.0)
                eps_fb_att    = self._draw_event_counts("EPS Fallback Voice",         _load_cls, _ues_cls, _amf_n, 1.0)
                sms_att       = self._draw_event_counts("SMS",                        _load_cls, _ues_cls, _amf_n, 1.0)
                pdu_estab_succ = int(pdu_estab_att * self._noisy_succ(0.9970 * afact["succ"]))
                pdu_rel_succ   = int(pdu_rel_att   * self._noisy_succ(0.9990 * afact["succ"]))
                pdu_mod_succ   = int(pdu_mod_att   * self._noisy_succ(0.9960 * afact["succ"]))

                # ══ 5.2.7  AUTHENTICATION / SECURITY (AUTH) ═══════════════
                auth_att     = init_reg_att + inter_mob_reg_att + intra_mob_reg_att
                nas_sec_att  = auth_att
                auth_succ    = int(auth_att * self._noisy_succ(0.9985 * afact["succ"]))
                auth_fail    = auth_att - auth_succ
                nas_sec_succ = int(nas_sec_att * self._noisy_succ(0.9990 * afact["succ"]))

                # Per-slice auth count (proportional)
                auth_cls = {cls: max(0, int(auth_att * _svc_mix[cls])) for cls in _svc_mix}

                # ══ 5.2.8  N1/N2 INTERFACE LOAD (N1N2) ════════════════════
                n1n2_total = 0.0
                for ev_key, ev_dict in EVENT_DATA.items():
                    for cls, n_ues_c in _ues_cls.items():
                        if n_ues_c <= 0:
                            continue
                        scale_c = SLICE_PROC_SCALE.get(cls, {}).get(ev_key, 1.0)
                        mean_ev = (ev_dict["bh_rate"] * scale_c * n_ues_c
                                   * STEP_MIN / 60.0) / _amf_n
                        mean_ev *= _load_cls.get(cls, load) * anom_req
                        n1n2_total += mean_ev * ev_dict["msgs"]

                n1_msgs_sent = int(StatUtils.sample_nb(n1n2_total * 0.50, 40, amf_rng))
                n1_msgs_recv = int(StatUtils.sample_nb(n1n2_total * 0.50 * amf_rng.uniform(0.95, 1.05), 40, amf_rng))
                n2_msgs_sent = int(StatUtils.sample_nb(n1n2_total * 0.50, 40, amf_rng))
                n2_msgs_recv = int(StatUtils.sample_nb(n1n2_total * 0.50 * amf_rng.uniform(0.95, 1.05), 40, amf_rng))
                ngap_active  = int(amf_rng.uniform(48, 56))

                # ══ 5.2.9  RESOURCE / PERFORMANCE (RES) ══════════════════

                # ── Aggregate M/M/c queueing model (all slices combined) ──
                total_events = (total_reg_att + total_ho_att + paging_att
                                + pdu_estab_att + n2_rel_att)
                lam_s    = max(total_events / self._SLOT_S, 1e-6)
                mu_s     = self._MU_SRV * vcpus * inst_cap
                _Ts, Wq, rho_mmc = StatUtils.mmc_sojourn(lam_s, mu_s, vcpus)
                T_QUEUE_ms = Wq * 1000.0 * afact["lat"]

                # ── Jackson network per-slice throughput ──────────────────
                lam_per_cls = {
                    cls: max(
                        sum(self._slice_mean(ek, cls, _ues_cls[cls], _amf_n,
                                             _load_cls[cls])
                            for ek in EVENT_DATA) / self._SLOT_S, 1e-9)
                    for cls in _svc_mix
                }
                jackson_tput = StatUtils.jackson_throughput(
                    lam_per_cls, mu_s, vcpus
                )

                # ── Per-procedure NAS latency ─────────────────────────────
                def _proc_lat(base_ms: float) -> float:
                    total = T_QUEUE_ms + base_ms * afact["lat"]
                    return float(StatUtils.sample_lognormal_ms(total, 0.30, amf_rng))

                lat_init_reg_ms  = _proc_lat(25.0)
                lat_mob_reg_ms   = _proc_lat(18.0)
                lat_per_reg_ms   = _proc_lat(8.0)
                lat_srv_req_ms   = _proc_lat(10.0)
                lat_n2_rel_ms    = _proc_lat(5.0)
                lat_xn_ho_ms     = _proc_lat(12.0)
                lat_n2_ho_ms     = _proc_lat(20.0)
                lat_pdu_estab_ms = _proc_lat(22.0)
                lat_auth_ms      = _proc_lat(15.0)
                lat_paging_ms    = _proc_lat(6.0)

                _lat_counts = [
                    (init_reg_att,                   lat_init_reg_ms),
                    (mob_reg_att,                    lat_mob_reg_ms),
                    (per_reg_att,                    lat_per_reg_ms),
                    (srv_req_att,                    lat_srv_req_ms),
                    (n2_rel_att,                     lat_n2_rel_ms),
                    (intra_xn_ho_att,                lat_xn_ho_ms),
                    (intra_n2_ho_att+inter_n2_ho_att,lat_n2_ho_ms),
                    (pdu_estab_att,                  lat_pdu_estab_ms),
                    (auth_att,                       lat_auth_ms),
                    (paging_att,                     lat_paging_ms),
                ]
                _total_cnt = sum(c for c, _ in _lat_counts)
                lat_ms = (sum(c * l for c, l in _lat_counts) / _total_cnt
                          if _total_cnt > 0 else T_QUEUE_ms + 10.0)

                # ── Aggregate CPU model (procedure-weighted + M/M/c blend) ─
                # Weighted by per-slice CPU multiplier (SLICE_CPU_MULT) so that
                # changing eMBB/mMTC/URLLC proportions is correctly reflected.
                # Each event count is split proportionally by _svc_mix, then
                # multiplied by the slice CPU multiplier before summing.
                def _cpu_w_slice(att_count: int, ev_key: str) -> float:
                    w = EVENT_DATA[ev_key]["cpu_w"]
                    return sum(
                        att_count * _svc_mix[cls] * w * SLICE_CPU_MULT[cls]
                        for cls in _svc_mix
                    )

                cpu_equiv = (
                    _cpu_w_slice(init_reg_att,                          "Initial Registration")     +
                    _cpu_w_slice(mob_reg_att,                           "Inter-AMF Mobility Reg")   +
                    _cpu_w_slice(per_reg_att,                           "Periodic Registration")    +
                    _cpu_w_slice(srv_req_att,                           "Service Request")          +
                    _cpu_w_slice(n2_rel_att,                            "N2 Release")               +
                    _cpu_w_slice(intra_xn_ho_att,                       "Intra-AMF Xn Handover")   +
                    _cpu_w_slice(intra_n2_ho_att + inter_n2_ho_att,     "Inter-AMF N2 Handover")   +
                    _cpu_w_slice(pdu_estab_att,                         "PDU Session Establishment")+
                    _cpu_w_slice(auth_att,                              "Initial Registration")     +
                    _cpu_w_slice(paging_att,                            "PS Paging")
                )
                # Rate-based CPU rho — slot-size independent
                lam_cpu_eff   = cpu_equiv / _SLOT_S_eff           # events/s
                vcpu_cap_slot = self._MU_SRV * vcpus * inst_cap   # events/s capacity
                rho_cpu       = lam_cpu_eff / max(vcpu_cap_slot, 1e-9)
                rho_blend     = 0.65 * rho_cpu + 0.35 * rho_mmc
                cpu_base      = min(100.0, rho_blend * 100.0 * afact["cpu"])
                cpu_util      = float(np.clip(cpu_base + amf_rng.normal(0.0, 1.5), 0.5, 99.5))

                # ── Aggregate memory model ────────────────────────────────
                active_pdu_sess = connected * 2.5
                queue_len_raw   = max(0.0, lam_s * Wq * self._SLOT_S)
                mem_base_mb     = 512.0
                mem_ctx_mb      = 0.005 * active_ctx
                mem_pdu_mb      = 0.001 * active_pdu_sess
                mem_sigbuf_mb   = 0.04  * queue_len_raw
                mem_auth_mb     = min(180.0, 0.08 * auth_att)
                mem_total_raw   = mem_base_mb + mem_ctx_mb + mem_pdu_mb + mem_sigbuf_mb + mem_auth_mb
                mem_total_mb    = mem_total_raw * afact["mem"]
                mem_util        = float(np.clip(
                    100.0 * mem_total_mb / self.mem_max_mb + amf_rng.normal(0.0, 0.6), 1.0, 99.5
                ))

                # ── v14 NEW: Per-slice CPU, Memory, Latency ──────────────
                # Build per-slice event-count dicts for CPU computation
                slice_cpu_pct: Dict[str, float] = {}
                slice_mem_mb:  Dict[str, float] = {}
                slice_lat_ms:  Dict[str, float] = {}

                for cls in _svc_mix:
                    # Per-slice event counts (proportional slice_mean weighting)
                    ev_counts_cls: Dict[str, int] = {}
                    for ev_key in EVENT_DATA:
                        lam_cls_ev = self._slice_mean(ev_key, cls, _ues_cls[cls], _amf_n, _load_cls[cls])
                        ev_counts_cls[ev_key] = max(0, int(StatUtils.sample_nb(
                            lam_cls_ev * anom_req, EVENT_DATA[ev_key]["nb_k"], amf_rng
                        )))
                    slice_cpu_pct[cls] = self._compute_slice_cpu(
                        ev_counts_cls, vcpus, inst_cap, afact, cls
                    )
                    slice_mem_mb[cls]  = self._compute_slice_mem(
                        ues_conn_cls[cls], auth_cls[cls], queue_len_raw, afact, cls
                    )
                    slice_lat_ms[cls]  = self._compute_slice_lat(Wq, afact, cls, amf_rng)

                # CPU cycles per message (efficiency metric)
                total_msgs        = max(n1_msgs_sent + n2_msgs_sent, 1)
                cpu_cycles_per_msg = (cpu_equiv * 1e6) / total_msgs   # approx instruction equiv

                # Memory bytes per UE
                mem_bytes_per_ue = (mem_total_mb * 1024 * 1024) / max(active_ctx, 1)

                # Slice load entropy
                slice_entropy = self._slice_entropy(_load_cls)

                throughput_mps = (n1_msgs_sent + n2_msgs_sent) / self._SLOT_S
                queue_len      = int(max(0, amf_rng.poisson(max(queue_len_raw, 1e-6))))
                active_workers = int(np.clip(rho_mmc * vcpus * 10, 0, vcpus * 10))

                # ─── Assemble row (3GPP TS 28.552 naming conventions) ────
                row = {
                    # ── Metadata ────────────────────────────────────────
                    "timestamp":          ts,
                    "amf_instance_id":    inst_id,
                    "is_anomaly":         int(is_anom),
                    "anomaly_type":       anom_type,
                    "anomaly_intensity":  sc.get("intensity", "none") if is_anom else "none",
                    "anomaly_sigmoid_w":  round(sig_w, 4),
                    "composite_load":     round(float(load), 4),
                    "rho":                round(float(rho_mmc), 4),

                    # ── 5.2.1 Registration Management ──────────────────
                    "RM.RegReqAtt":          total_reg_att,
                    "RM.RegReqSucc":         total_reg_succ,
                    "RM.RegReqFail":         total_reg_att - total_reg_succ,
                    "RM.RegSuccRate":        round(100.0 * total_reg_succ / max(total_reg_att,1), 3),
                    "RM.InitRegReqAtt":      init_reg_att,
                    "RM.InitRegReqSucc":     init_reg_succ,
                    "RM.MobilityRegReqAtt":  mob_reg_att,
                    "RM.MobilityRegReqSucc": mob_reg_succ,
                    "RM.PeriodicRegReqAtt":  per_reg_att,
                    "RM.PeriodicRegReqSucc": per_reg_succ,
                    "RM.DeregReqAtt":        dereg_att,
                    "RM.DeregReqSucc":       dereg_succ,

                    # ── 5.2.2 Connection Management ────────────────────
                    "CM.ServiceReqAtt":      srv_req_att,
                    "CM.ServiceReqSucc":     srv_req_succ,
                    "CM.ServiceReqSuccRate": round(100.0 * srv_req_succ / max(srv_req_att,1), 3),
                    "CM.N2RelAtt":           n2_rel_att,
                    "CM.N2RelSucc":          n2_rel_succ,

                    # ── 5.2.3 Mobility Management ──────────────────────
                    "MM.HoReqAtt":           total_ho_att,
                    "MM.HoReqSucc":          total_ho_succ,
                    "MM.HoSuccRate":         round(100.0 * total_ho_succ / max(total_ho_att,1), 3),
                    "MM.XnHoReqAtt":         intra_xn_ho_att,
                    "MM.XnHoReqSucc":        intra_xn_ho_succ,
                    "MM.N2IntraHoReqAtt":    intra_n2_ho_att,
                    "MM.N2IntraHoReqSucc":   intra_n2_ho_succ,
                    "MM.N2InterHoReqAtt":    inter_n2_ho_att,
                    "MM.N2InterHoReqSucc":   inter_n2_ho_succ,
                    "MM.EPS2fiveGSMobAtt":   eps5g_mob,
                    "MM.fiveGS2EPSMobAtt":   five_eps_mob,
                    "MM.N26HoAtt":           inter_n26_ho,

                    # ── 5.2.4 Paging ───────────────────────────────────
                    "PAG.PagingReqAtt":      paging_att,
                    "PAG.PagingReqSucc":     paging_succ,
                    "PAG.PagingSuccRate":    round(100.0 * paging_succ / max(paging_att,1), 3),
                    "PAG.PagingDiscarded":   paging_disc,
                    "PAG.PagingRetry":       paging_retry,
                    "PAG.UeInIdleMode":      int(idle),

                    # ── 5.2.5 UE Context ────────────────────────────────
                    "UC.UeContextCreated":   ctx_created,
                    "UC.UeContextReleased":  ctx_released,
                    "UC.UeContextModified":  ctx_modified,
                    "UC.ActiveUeContext":    active_ctx,
                    "UC.MaxUeContextCap":    max_ctx_cap,
                    "UC.ContextUtilRate":    round(min(100.0, 100.0 * active_ctx / max(max_ctx_cap,1)), 3),

                    # ── 5.2.6 PDU Session ───────────────────────────────
                    "SM.PduSessEstabAtt":       pdu_estab_att,
                    "SM.PduSessEstabSucc":      pdu_estab_succ,
                    "SM.PduSessEstabSuccRate":  round(100.0 * pdu_estab_succ / max(pdu_estab_att,1), 3),
                    "SM.PduSessRelAtt":         pdu_rel_att,
                    "SM.PduSessRelSucc":        pdu_rel_succ,
                    "SM.PduSessModAtt":         pdu_mod_att,
                    "SM.PduSessModSucc":        pdu_mod_succ,
                    "SM.VoNRAtt":               vonr_att,
                    "SM.EPSFallbackAtt":        eps_fb_att,
                    "SM.SMSAtt":                sms_att,

                    # ── 5.2.7 Authentication ────────────────────────────
                    "AUTH.AuthProcAtt":         auth_att,
                    "AUTH.AuthProcSucc":        auth_succ,
                    "AUTH.AuthProcFail":        auth_fail,
                    "AUTH.AuthSuccRate":        round(100.0 * auth_succ / max(auth_att,1), 3),
                    "AUTH.NasSecModeAtt":       nas_sec_att,
                    "AUTH.NasSecModeSucc":      nas_sec_succ,

                    # ── 5.2.8 N1/N2 Interface Load ───────────────────────
                    "N1N2.N1MsgSent":           n1_msgs_sent,
                    "N1N2.N1MsgRecv":           n1_msgs_recv,
                    "N1N2.N2MsgSent":           n2_msgs_sent,
                    "N1N2.N2MsgRecv":           n2_msgs_recv,
                    "N1N2.NgapConnActive":       ngap_active,
                    "N1N2.TotalMsgLoad":         int(n1n2_total),

                    # ── 5.2.9 Resource / Performance (aggregate) ─────────
                    "RES.CpuUtil":              round(cpu_util, 2),
                    "RES.RhoCPU_weighted":      round(float(min(rho_blend, 1.5)), 4),
                    "RES.MemUtil":              round(mem_util, 2),
                    "RES.MemTotal_MB":          round(mem_total_mb, 1),
                    "RES.MemBase_MB":           round(mem_base_mb, 1),
                    "RES.MemCtx_MB":            round(mem_ctx_mb, 1),
                    "RES.MemPDU_MB":            round(mem_pdu_mb, 1),
                    "RES.MemSigBuf_MB":         round(mem_sigbuf_mb, 2),
                    "RES.MemAuthCache_MB":      round(mem_auth_mb, 1),
                    "RES.Latency_ms":           round(lat_ms, 3),
                    "RES.Wq_ms":                round(T_QUEUE_ms, 3),
                    "RES.Lat_InitReg_ms":       round(lat_init_reg_ms, 3),
                    "RES.Lat_MobReg_ms":        round(lat_mob_reg_ms, 3),
                    "RES.Lat_PerReg_ms":        round(lat_per_reg_ms, 3),
                    "RES.Lat_SrvReq_ms":        round(lat_srv_req_ms, 3),
                    "RES.Lat_N2Rel_ms":         round(lat_n2_rel_ms, 3),
                    "RES.Lat_XnHO_ms":          round(lat_xn_ho_ms, 3),
                    "RES.Lat_N2HO_ms":          round(lat_n2_ho_ms, 3),
                    "RES.Lat_PduEstab_ms":      round(lat_pdu_estab_ms, 3),
                    "RES.Lat_Auth_ms":          round(lat_auth_ms, 3),
                    "RES.Lat_Paging_ms":        round(lat_paging_ms, 3),
                    "RES.Throughput_mps":       round(throughput_mps, 3),
                    "RES.QueueLength":          queue_len,
                    "RES.ActiveWorkers":        active_workers,
                    "RES.ActiveUEs":            active_ues,
                    "RES.ConnectedUEs":         connected,
                    "RES.IdleUEs":              int(idle),

                    # ── v14 NEW: Per-Slice Resource KPIs ────────────────
                    "RES.CPU_eMBB_pct":         round(slice_cpu_pct["eMBB"],  2),
                    "RES.CPU_mMTC_pct":         round(slice_cpu_pct["mMTC"],  2),
                    "RES.CPU_URLLC_pct":        round(slice_cpu_pct["URLLC"], 2),
                    "RES.Mem_eMBB_MB":          round(slice_mem_mb["eMBB"],   2),
                    "RES.Mem_mMTC_MB":          round(slice_mem_mb["mMTC"],   2),
                    "RES.Mem_URLLC_MB":         round(slice_mem_mb["URLLC"],  2),
                    "RES.Lat_eMBB_ms":          round(slice_lat_ms["eMBB"],   3),
                    "RES.Lat_mMTC_ms":          round(slice_lat_ms["mMTC"],   3),
                    "RES.Lat_URLLC_ms":         round(slice_lat_ms["URLLC"],  3),
                    "RES.CpuCyclesPerMsg":       round(cpu_cycles_per_msg, 1),
                    "RES.MemBytesPerUE":         round(mem_bytes_per_ue, 1),
                    "RES.SliceLoadEntropy":      round(slice_entropy, 4),
                    # Jackson per-class throughput
                    "RES.Jackson_eMBB_eps":      round(jackson_tput["eMBB"],  3),
                    "RES.Jackson_mMTC_eps":      round(jackson_tput["mMTC"],  3),
                    "RES.Jackson_URLLC_eps":     round(jackson_tput["URLLC"], 3),
                    # SLA breach flags (bool int)
                    "SLA.eMBB_breach":           int(slice_lat_ms["eMBB"]  > SLICE_LAT_PARAMS["eMBB"]["sla_ms"]),
                    "SLA.mMTC_breach":           int(slice_lat_ms["mMTC"]  > SLICE_LAT_PARAMS["mMTC"]["sla_ms"]),
                    "SLA.URLLC_breach":          int(slice_lat_ms["URLLC"] > SLICE_LAT_PARAMS["URLLC"]["sla_ms"]),
                }
                all_rows.append(row)

            # Report progress after each AMF instance
            if progress_callback is not None:
                progress_callback((amf_idx + 1) / _amf_n * 100.0)

        df = pd.DataFrame(all_rows)
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        return df

# ---

# ──────────────────────────────────────────────────────────────────────
# Section 7 — Dataset Validation
# ──────────────────────────────────────────────────────────────────────

---
## Section 6 — Dataset Validation


In [11]:
def validate_dataset(df: pd.DataFrame, amf_instances: int = 1) -> dict:
    """
    Structural checks run against the *internal* (116-column) DataFrame
    immediately after generation.

    Returns a dict mapping check name → bool.  All must be True before export.

    Checks
    ------
    row_count_correct   : will be set by the caller after computing expected rows
    no_nan_in_key_cols  : core label + resource columns contain no NaN
    rates_in_range      : success-rate columns are in [0, 100]
    counts_non_negative : all count columns are >= 0
    anomaly_labels_ok   : is_anomaly is 0/1; anomaly_type is non-null string
    correct_amf_instances : number of unique AMF instance IDs matches amf_instances
    timestamps_ordered  : timestamp column is strictly monotonically increasing
    cpu_in_range        : RES.CpuUtil values are in [0, 100]
    mem_in_range        : RES.MemUtil values are in [0, 100]
    succ_le_att         : RM.RegReqSucc <= RM.RegReqAtt for every row
    """
    results = {}

    # ── row_count_correct ────────────────────────────────────────────────────
    # Placeholder — caller sets this after computing (duration * 60 // step) * instances
    results["row_count_correct"] = True

    # ── no_nan_in_key_cols ───────────────────────────────────────────────────
    key_cols = [c for c in [
        "timestamp", "amf_instance_id", "is_anomaly", "anomaly_type",
        "RES.CpuUtil", "RES.MemUtil", "RES.Latency_ms",
        "RM.RegReqAtt", "RM.RegReqSucc",
    ] if c in df.columns]
    results["no_nan_in_key_cols"] = bool(df[key_cols].notna().all().all())

    # ── rates_in_range ───────────────────────────────────────────────────────
    rate_cols = [c for c in df.columns if "Rate" in c or "Util" in c]
    if rate_cols:
        results["rates_in_range"] = bool(
            (df[rate_cols] >= 0).all().all() and (df[rate_cols] <= 100).all().all()
        )
    else:
        results["rates_in_range"] = True

    # ── counts_non_negative ──────────────────────────────────────────────────
    count_cols = [c for c in df.columns
                  if any(t in c for t in ["Att", "Succ", "Fail", "Req", "Msg"])]
    if count_cols:
        results["counts_non_negative"] = bool((df[count_cols] >= 0).all().all())
    else:
        results["counts_non_negative"] = True

    # ── anomaly_labels_ok ────────────────────────────────────────────────────
    results["anomaly_labels_ok"] = bool(
        df["is_anomaly"].isin([0, 1]).all()
        and df["anomaly_type"].notna().all()
    )

    # ── correct_amf_instances ────────────────────────────────────────────────
    results["correct_amf_instances"] = (
        df["amf_instance_id"].nunique() == amf_instances
    )

    # ── timestamps_ordered ───────────────────────────────────────────────────
    ts = pd.to_datetime(df["timestamp"])
    results["timestamps_ordered"] = bool(ts.is_monotonic_increasing)

    # ── cpu_in_range ─────────────────────────────────────────────────────────
    if "RES.CpuUtil" in df.columns:
        results["cpu_in_range"] = bool(
            (df["RES.CpuUtil"] >= 0).all() and (df["RES.CpuUtil"] <= 100).all()
        )
    else:
        results["cpu_in_range"] = True

    # ── mem_in_range ─────────────────────────────────────────────────────────
    if "RES.MemUtil" in df.columns:
        results["mem_in_range"] = bool(
            (df["RES.MemUtil"] >= 0).all() and (df["RES.MemUtil"] <= 100).all()
        )
    else:
        results["mem_in_range"] = True

    # ── succ_le_att ──────────────────────────────────────────────────────────
    if "RM.RegReqSucc" in df.columns and "RM.RegReqAtt" in df.columns:
        results["succ_le_att"] = bool(
            (df["RM.RegReqSucc"] <= df["RM.RegReqAtt"]).all()
        )
    else:
        results["succ_le_att"] = True

    return results


print("✓ validate_dataset() defined.")


✓ validate_dataset() defined.


---
## Section 7 — Dataset Validation

---
## Section 8 — Export Utilities  (CSV · JSON · ZIP)

In [12]:
def export_dataset(df: pd.DataFrame, base_path: str) -> Tuple[str, str]:
    # ── Strip internal generator metadata columns before export ──────────
    # These columns are used internally by the generator/anomaly engine and
    # must NOT appear in the public dataset — they would cause data leakage
    # in any ML model trained on the dataset:
    #   anomaly_sigmoid_w : = 0.0 for ALL normal rows, > 0 for ALL anomaly rows
    #                         → perfect label proxy, not a real AMF KPI
    #   anomaly_intensity : string label "none"/"moderate"/etc.
    #                         → direct anomaly metadata, not a KPI
    #   composite_load    : internal load scalar driving the anomaly engine
    #   rho               : raw M/M/c utilisation before noise — internal calc
    _INTERNAL_COLS = {
        "anomaly_sigmoid_w",
        "anomaly_intensity",
        "composite_load",
        "rho",
    }
    export_cols = [c for c in df.columns if c not in _INTERNAL_COLS]
    df_export   = df[export_cols]

    csv_p  = base_path + ".csv"
    json_p = base_path + ".json"
    df_export.to_csv(csv_p, index=False)
    df_export.to_json(json_p, orient="records", date_format="iso", indent=2)
    print(f"  CSV  → {csv_p}  ({len(export_cols)} columns, "
          f"{len(_INTERNAL_COLS & set(df.columns))} internal cols stripped)")
    print(f"  JSON → {json_p}")
    return csv_p, json_p


def write_metadata(df: pd.DataFrame, path: str, config: Dict) -> None:
    meta = {
        "generator":       "amf_synthetic_dataset_v14",
        "3gpp_standard":   "TS 28.552 v19.6.0",
        "generated_at":    datetime.utcnow().isoformat(),
        "config":          {k: str(v) for k, v in config.items()},
        "duration_hours":  DURATION_HOURS,
        "step_min":        self.step_min if hasattr(self, "step_min") else STEP_MIN,
        "total_rows":      len(df),
        "columns":         list(df.columns),
        "anomaly_rows":    int(df["is_anomaly"].sum()),
        "sla_breach_eMBB": int(df["SLA.eMBB_breach"].sum()),
        "sla_breach_mMTC": int(df["SLA.mMTC_breach"].sum()),
        "sla_breach_URLLC":int(df["SLA.URLLC_breach"].sum()),
        "stats": {c: {"mean": round(float(df[c].mean()),4),
                       "std":  round(float(df[c].std()),4),
                       "p95":  round(float(df[c].quantile(0.95)),4)}
                  for c in ["RES.CpuUtil","RES.MemUtil","RES.Latency_ms",
                              "RES.Lat_eMBB_ms","RES.Lat_mMTC_ms","RES.Lat_URLLC_ms"]
                  if c in df.columns},
    }
    with open(path, "w") as f:
        json.dump(meta, f, indent=2)
    print(f"  Metadata → {path}")

---
## Section 9 — Plotting — Shared Helpers + Per-Section §5.2.1 – §5.2.9

In [13]:
_TS_FMT = mdates.DateFormatter("%m/%d")
_TS_LOC = mdates.DayLocator(interval=2)
SLICE_COLORS = {"eMBB": "#2196F3", "mMTC": "#4CAF50", "URLLC": "#FF5722"}
_PALETTE = ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b","#e377c2","#7f7f7f"]


### `def _setup_ax`

In [14]:
def _setup_ax(ax, title: str, ylabel: str = "", ts_fmt: str = "%m/%d",
              interval_days: int = 2) -> None:
    ax.set_title(title, fontweight="bold", fontsize=10)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=9)
    ax.set_xlabel("Date", fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter(ts_fmt))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=interval_days))
    ax.tick_params(axis="x", labelrotation=30, labelsize=8)
    ax.tick_params(axis="y", labelsize=8)
    ax.grid(True, linestyle="--", alpha=0.4)


### `def _shade_anomalies`

In [15]:
def _shade_anomalies(ax, _agg_ts, df_full: pd.DataFrame) -> None:
    """Shade anomaly windows on a time-series axis, coloured by anomaly type."""
    type_colors = {
        "cpu_overload":            "#e53935",
        "ddos_fake_registrations": "#8e24aa",
        "memory_leak":             "#fb8c00",
        "registration_storm":      "#f4511e",
        "handover_failure":        "#43a047",
        "signaling_storm":         "#1e88e5",
        "paging_flood":            "#00acc1",
        "nas_replay_attack":       "#6d4c41",
        "slice_isolation_failure": "#c62828",
        "amf_overload_cascade":    "#4a148c",
    }
    labeled: set = set()
    anoms = df_full[df_full["is_anomaly"] == 1][["timestamp","anomaly_type"]].drop_duplicates()
    if anoms.empty:
        return
    anoms = anoms.sort_values("timestamp")
    anoms["gap"] = anoms["timestamp"].diff() > pd.Timedelta(minutes=STEP_MIN * 2)
    anoms["grp"] = anoms["gap"].cumsum()
    for (atype, grp), sub in anoms.groupby(["anomaly_type","grp"]):
        t0 = sub["timestamp"].min()
        t1 = sub["timestamp"].max() + pd.Timedelta(minutes=STEP_MIN)
        col = type_colors.get(atype, "#bdbdbd")
        lbl = atype if atype not in labeled else "_nolegend_"
        ax.axvspan(t0, t1, color=col, alpha=0.18, label=lbl, zorder=0)
        labeled.add(atype)


### `def _resample_agg`

In [16]:
def _resample_agg(df: pd.DataFrame, cols: List[str],
                   func: str = "sum", rule: str = "1h") -> pd.DataFrame:
    """Aggregate cols across all AMF instances then resample."""
    avail = [c for c in cols if c in df.columns]
    if not avail:
        return pd.DataFrame()
    grouped = df.groupby("timestamp")[avail].agg("mean" if func == "mean" else func)
    grouped.index = pd.to_datetime(grouped.index)
    return grouped.resample(rule).mean()


### `def stats_qq_lognormal`

In [17]:
def stats_qq_lognormal(data: np.ndarray) -> Tuple:
    """Compute QQ data points for a Log-Normal fit."""
    from scipy import stats as sp_stats
    log_data = np.log(data[data > 0])
    mu, sigma = log_data.mean(), log_data.std()
    sorted_data = np.sort(data[data > 0])
    n = len(sorted_data)
    probs = (np.arange(1, n + 1) - 0.5) / n
    theoretical = np.exp(sp_stats.norm.ppf(probs) * sigma + mu)
    slope, intercept, r, _, _ = sp_stats.linregress(theoretical, sorted_data)
    return (theoretical, sorted_data), (slope, intercept, r)


# ── Section 5.2.1 ──────────────────────────────────────────────────────────


### `def plot_521_registration`

In [18]:
def plot_521_registration(df: pd.DataFrame, out_dir: str) -> str:
    fig, axes = plt.subplots(4, 1, figsize=(15, 18), dpi=130,
                              gridspec_kw={"hspace": 0.55})
    fig.suptitle("3GPP TS 28.552 §5.2.1 – Registration Management KPIs",
                 fontsize=13, fontweight="bold")
    df = df.copy(); df["timestamp"] = pd.to_datetime(df["timestamp"])
    rule = "1h"

    ax = axes[0]
    for i, inst in enumerate(sorted(df["amf_instance_id"].unique())):
        sub = (df[df["amf_instance_id"] == inst]
               .set_index("timestamp")[["RM.RegReqAtt","RM.RegReqFail"]]
               .resample(rule).sum())
        ax.plot(sub.index, sub["RM.RegReqAtt"], color=_PALETTE[i % 8],
                linewidth=1.2, label=f"{inst} Att", alpha=0.9)
        ax.fill_between(sub.index, sub["RM.RegReqFail"], alpha=0.2,
                        color=_PALETTE[i % 8])
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "RM.RegReqAtt / RegReqFail per AMF Instance (1 h)", "Registrations / h")
    ax.legend(ncol=5, fontsize=7, loc="upper left")

    ax = axes[1]
    rs = _resample_agg(df, ["RM.InitRegReqAtt","RM.MobilityRegReqAtt","RM.PeriodicRegReqAtt"], "sum", rule)
    bot = np.zeros(len(rs))
    for col, lbl, c in zip(["RM.InitRegReqAtt","RM.MobilityRegReqAtt","RM.PeriodicRegReqAtt"],
                             ["Initial","Mobility","Periodic"],
                             ["#1f77b4","#ff7f0e","#2ca02c"]):
        if col in rs.columns:
            ax.fill_between(rs.index, bot, bot + rs[col].values, alpha=0.65, color=c, label=lbl)
            bot += rs[col].fillna(0).values
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "RM – Registration Type Breakdown", "Registrations / h")
    ax.legend(fontsize=8)

    ax = axes[2]; ax2 = ax.twinx()
    sr = _resample_agg(df, ["RM.RegSuccRate"], "mean", rule)
    drg = _resample_agg(df, ["RM.DeregReqAtt","RM.DeregReqSucc"], "sum", rule)
    ax.plot(sr.index, sr["RM.RegSuccRate"], color="#d62728", linewidth=1.5, label="RegSuccRate (%)")
    ax.set_ylim(85, 101)
    if "RM.DeregReqAtt" in drg.columns:
        ax2.bar(drg.index, drg["RM.DeregReqAtt"], width=1/24, alpha=0.3,
                color="#9467bd", label="DeregAtt")
        ax2.plot(drg.index, drg["RM.DeregReqSucc"], color="#8c564b",
                 linewidth=0.9, label="DeregSucc")
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "RM – Registration Success Rate & Deregistration", "Succ Rate (%)")
    ax2.set_ylabel("Deregistrations / h", fontsize=9)
    ax.legend(loc="lower left", fontsize=8); ax2.legend(loc="lower right", fontsize=8)

    ax = axes[3]
    df["_hr"] = df["timestamp"].dt.hour
    bp = ax.boxplot([df[df["_hr"]==h]["RM.RegReqAtt"].values for h in range(24)],
                    positions=range(24), showfliers=False, patch_artist=True,
                    medianprops={"color":"red","linewidth":1.5},
                    boxprops={"facecolor":"#aec6e8","alpha":0.75})
    ax.set_title("RM – Diurnal Distribution of RM.RegReqAtt", fontweight="bold", fontsize=10)
    ax.set_xlabel("Hour of Day", fontsize=9); ax.set_ylabel("Reg Requests / 15-min slot", fontsize=9)
    ax.set_xticks(range(0, 24, 2)); ax.grid(True, linestyle="--", alpha=0.4)
    df.drop(columns=["_hr"], inplace=True)

    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "sec521_registration_management.png")
    fig.savefig(path, bbox_inches="tight", dpi=130); plt.close(fig)
    print(f"  §5.2.1  → {path}"); return path


# ── Section 5.2.2 ──────────────────────────────────────────────────────────


### `def plot_522_connection`

In [19]:
def plot_522_connection(df: pd.DataFrame, out_dir: str) -> str:
    fig, axes = plt.subplots(3, 1, figsize=(15, 13), dpi=130,
                              gridspec_kw={"hspace": 0.55})
    fig.suptitle("3GPP TS 28.552 §5.2.2 – Connection Management KPIs",
                 fontsize=13, fontweight="bold")
    df = df.copy(); df["timestamp"] = pd.to_datetime(df["timestamp"])
    rule = "1h"

    ax = axes[0]
    for i, inst in enumerate(sorted(df["amf_instance_id"].unique())):
        sub = (df[df["amf_instance_id"]==inst]
               .set_index("timestamp")[["CM.ServiceReqAtt","CM.ServiceReqSucc"]]
               .resample(rule).sum())
        ax.plot(sub.index, sub["CM.ServiceReqAtt"], color=_PALETTE[i%8], linewidth=1.2, label=f"{inst} Att")
        ax.plot(sub.index, sub["CM.ServiceReqSucc"], color=_PALETTE[i%8], linewidth=0.8, ls="--", alpha=0.7)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "CM.ServiceReqAtt / ServiceReqSucc (solid=Att, dashed=Succ)", "Svc Req / h")
    ax.legend(ncol=5, fontsize=7)

    ax = axes[1]
    sr = _resample_agg(df, ["CM.ServiceReqSuccRate"], "mean", rule)
    ax.plot(sr.index, sr["CM.ServiceReqSuccRate"], color="#d62728", linewidth=1.5)
    ax.axhline(99.0, color="orange", ls="--", lw=1.0, label="99 % threshold")
    ax.set_ylim(85, 101)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "CM.ServiceReqSuccRate (%)", "Success Rate (%)")
    ax.legend(fontsize=8)

    ax = axes[2]
    n2 = _resample_agg(df, ["CM.N2RelAtt","CM.N2RelSucc"], "sum", rule)
    if "CM.N2RelAtt" in n2.columns:
        ax.fill_between(n2.index, n2["CM.N2RelAtt"], alpha=0.4, color="#2ca02c", label="N2RelAtt")
        ax.plot(n2.index, n2["CM.N2RelSucc"], color="#1f77b4", linewidth=1.2, label="N2RelSucc")
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "CM.N2RelAtt / N2RelSucc", "Count / h")
    ax.legend(fontsize=8)

    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "sec522_connection_management.png")
    fig.savefig(path, bbox_inches="tight", dpi=130); plt.close(fig)
    print(f"  §5.2.2  → {path}"); return path


# ── Section 5.2.3 ──────────────────────────────────────────────────────────


### `def plot_523_mobility`

In [20]:
def plot_523_mobility(df: pd.DataFrame, out_dir: str) -> str:
    fig, axes = plt.subplots(4, 1, figsize=(15, 18), dpi=130,
                              gridspec_kw={"hspace": 0.55})
    fig.suptitle("3GPP TS 28.552 §5.2.3 – Mobility Management KPIs",
                 fontsize=13, fontweight="bold")
    df = df.copy(); df["timestamp"] = pd.to_datetime(df["timestamp"])
    rule = "1h"

    ax = axes[0]
    for i, inst in enumerate(sorted(df["amf_instance_id"].unique())):
        sub = (df[df["amf_instance_id"]==inst]
               .set_index("timestamp")[["MM.HoReqAtt","MM.HoReqSucc"]]
               .resample(rule).sum())
        ax.plot(sub.index, sub["MM.HoReqAtt"], color=_PALETTE[i%8], linewidth=1.2, label=f"{inst} Att")
        ax.plot(sub.index, sub["MM.HoReqSucc"], color=_PALETTE[i%8], linewidth=0.8, ls="--", alpha=0.7)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "MM.HoReqAtt / HoReqSucc (solid=Att, dashed=Succ)", "Handovers / h")
    ax.legend(ncol=5, fontsize=7)

    ax = axes[1]
    rs = _resample_agg(df, ["MM.XnHoReqAtt","MM.N2IntraHoReqAtt","MM.N2InterHoReqAtt"], "sum", rule)
    bot = np.zeros(len(rs))
    for col, lbl, c in zip(["MM.XnHoReqAtt","MM.N2IntraHoReqAtt","MM.N2InterHoReqAtt"],
                             ["Xn Intra","N2 Intra","N2 Inter"],
                             ["#1f77b4","#ff7f0e","#d62728"]):
        if col in rs.columns:
            ax.fill_between(rs.index, bot, bot + rs[col].values, alpha=0.65, color=c, label=lbl)
            bot += rs[col].fillna(0).values
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "MM – Handover Type Breakdown (Xn / N2-Intra / N2-Inter)", "Handovers / h")
    ax.legend(fontsize=8)

    ax = axes[2]
    sr = _resample_agg(df, ["MM.HoSuccRate"], "mean", rule)
    ax.plot(sr.index, sr["MM.HoSuccRate"], color="#d62728", linewidth=1.5, label="HoSuccRate (%)")
    ax.axhline(98.0, color="orange", ls="--", lw=1.0, label="98 % ref")
    ax.set_ylim(40, 102)
    for i, inst in enumerate(sorted(df["amf_instance_id"].unique())):
        sub = df[df["amf_instance_id"]==inst].set_index("timestamp")[["MM.HoSuccRate"]].resample(rule).mean()
        ax.plot(sub.index, sub["MM.HoSuccRate"], color=_PALETTE[i%8], lw=0.8, ls=":", alpha=0.7, label=inst)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "MM.HoSuccRate (%) – aggregate + per-instance", "Success Rate (%)")
    ax.legend(ncol=3, fontsize=7)

    ax = axes[3]
    mob = _resample_agg(df, ["MM.EPS2fiveGSMobAtt","MM.fiveGS2EPSMobAtt","MM.N26HoAtt"], "sum", rule)
    for col, lbl, c in zip(["MM.EPS2fiveGSMobAtt","MM.fiveGS2EPSMobAtt","MM.N26HoAtt"],
                             ["EPS→5GS Mobility","5GS→EPS Mobility","N26 HO"],
                             ["#2ca02c","#ff7f0e","#9467bd"]):
        if col in mob.columns:
            ax.plot(mob.index, mob[col], color=c, linewidth=1.2, label=lbl)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "MM – Inter-System Mobility & N26 HO Attempts", "Count / h")
    ax.legend(fontsize=8)

    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "sec523_mobility_management.png")
    fig.savefig(path, bbox_inches="tight", dpi=130); plt.close(fig)
    print(f"  §5.2.3  → {path}"); return path


# ── Section 5.2.4 ──────────────────────────────────────────────────────────


### `def plot_524_paging`

In [21]:
def plot_524_paging(df: pd.DataFrame, out_dir: str) -> str:
    fig, axes = plt.subplots(3, 1, figsize=(15, 13), dpi=130,
                              gridspec_kw={"hspace": 0.55})
    fig.suptitle("3GPP TS 28.552 §5.2.4 – Paging KPIs", fontsize=13, fontweight="bold")
    df = df.copy(); df["timestamp"] = pd.to_datetime(df["timestamp"])
    rule = "1h"

    ax = axes[0]
    agg_p = _resample_agg(df, ["PAG.PagingReqAtt","PAG.PagingReqSucc",
                                 "PAG.PagingDiscarded","PAG.PagingRetry"], "sum", rule)
    for col, lbl, c, ls in [("PAG.PagingReqAtt","PagingReqAtt","#1f77b4","fill"),
                              ("PAG.PagingReqSucc","PagingReqSucc","#2ca02c","-"),
                              ("PAG.PagingDiscarded","Discarded","#d62728","--"),
                              ("PAG.PagingRetry","Retry","#ff7f0e",":")]:
        if col in agg_p.columns:
            if ls == "fill":
                ax.fill_between(agg_p.index, agg_p[col], alpha=0.3, color=c, label=lbl)
            else:
                ax.plot(agg_p.index, agg_p[col], color=c, lw=1.1, ls=ls, label=lbl)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "PAG – Paging Att / Succ / Discarded / Retry (1 h)", "Pagings / h")
    ax.legend(fontsize=8, ncol=4)

    ax = axes[1]; ax2 = ax.twinx()
    sr = _resample_agg(df, ["PAG.PagingSuccRate"], "mean", rule)
    idle = _resample_agg(df, ["PAG.UeInIdleMode"], "sum", rule)
    ax.plot(sr.index, sr["PAG.PagingSuccRate"], color="#d62728", lw=1.5, label="PagingSuccRate (%)")
    ax.set_ylim(85, 101)
    if "PAG.UeInIdleMode" in idle.columns:
        ax2.fill_between(idle.index, idle["PAG.UeInIdleMode"], alpha=0.2,
                         color="#9467bd", label="UeInIdleMode")
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "PAG.PagingSuccRate (%) & UeInIdleMode", "Succ Rate (%)")
    ax2.set_ylabel("UEs in Idle Mode", fontsize=9)
    ax.legend(loc="lower left", fontsize=8); ax2.legend(loc="lower right", fontsize=8)

    ax = axes[2]
    df["_hr"] = df["timestamp"].dt.hour
    ax.boxplot([df[df["_hr"]==h]["PAG.PagingReqAtt"].values for h in range(24)],
               positions=range(24), showfliers=False, patch_artist=True,
               medianprops={"color":"red","linewidth":1.5},
               boxprops={"facecolor":"#b5d5f5","alpha":0.75})
    ax.set_title("PAG – Diurnal Distribution of PAG.PagingReqAtt", fontweight="bold", fontsize=10)
    ax.set_xlabel("Hour of Day", fontsize=9); ax.set_ylabel("Pagings / 15-min slot", fontsize=9)
    ax.set_xticks(range(0, 24, 2)); ax.grid(True, ls="--", alpha=0.4)
    df.drop(columns=["_hr"], inplace=True)

    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "sec524_paging.png")
    fig.savefig(path, bbox_inches="tight", dpi=130); plt.close(fig)
    print(f"  §5.2.4  → {path}"); return path


# ── Section 5.2.5 ──────────────────────────────────────────────────────────


### `def plot_525_ue_context`

In [22]:
def plot_525_ue_context(df: pd.DataFrame, out_dir: str) -> str:
    fig, axes = plt.subplots(3, 1, figsize=(15, 13), dpi=130,
                              gridspec_kw={"hspace": 0.55})
    fig.suptitle("3GPP TS 28.552 §5.2.5 – UE Context Management KPIs",
                 fontsize=13, fontweight="bold")
    df = df.copy(); df["timestamp"] = pd.to_datetime(df["timestamp"])
    rule = "1h"

    ax = axes[0]
    ctxs = _resample_agg(df, ["UC.UeContextCreated","UC.UeContextReleased","UC.UeContextModified"], "sum", rule)
    for col, lbl, c in zip(["UC.UeContextCreated","UC.UeContextReleased","UC.UeContextModified"],
                             ["Created","Released","Modified"],
                             ["#2ca02c","#d62728","#ff7f0e"]):
        if col in ctxs.columns:
            ax.plot(ctxs.index, ctxs[col], color=c, lw=1.3, label=lbl)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "UC – UE Context Created / Released / Modified", "UE Contexts / h")
    ax.legend(fontsize=8)

    ax = axes[1]
    for i, inst in enumerate(sorted(df["amf_instance_id"].unique())):
        sub = (df[df["amf_instance_id"]==inst]
               .set_index("timestamp")[["UC.ActiveUeContext"]].resample(rule).mean())
        ax.plot(sub.index, sub["UC.ActiveUeContext"], color=_PALETTE[i%8], lw=1.2, label=f"{inst}")
    cap = df.groupby("timestamp")["UC.MaxUeContextCap"].first()
    cap.index = pd.to_datetime(cap.index)
    ax.plot(cap.resample(rule).first().index, cap.resample(rule).first().values,
            color="black", lw=1.5, ls="--", label="Max Capacity")
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "UC.ActiveUeContext vs MaxUeContextCap per AMF", "UE Contexts")
    ax.legend(ncol=3, fontsize=7)

    ax = axes[2]
    for i, inst in enumerate(sorted(df["amf_instance_id"].unique())):
        sub = df[df["amf_instance_id"]==inst].set_index("timestamp")[["UC.ContextUtilRate"]].resample(rule).mean()
        ax.plot(sub.index, sub["UC.ContextUtilRate"], color=_PALETTE[i%8], lw=1.2, label=inst)
    ax.axhline(80, color="orange", ls="--", lw=1.0, label="80 % warning")
    ax.axhline(95, color="red", ls="--", lw=1.0, label="95 % critical")
    ax.set_ylim(0, 105)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "UC.ContextUtilRate (%) per AMF Instance", "Utilisation (%)")
    ax.legend(ncol=3, fontsize=7)

    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "sec525_ue_context.png")
    fig.savefig(path, bbox_inches="tight", dpi=130); plt.close(fig)
    print(f"  §5.2.5  → {path}"); return path


# ── Section 5.2.6 ──────────────────────────────────────────────────────────


### `def plot_526_pdu_session`

In [23]:
def plot_526_pdu_session(df: pd.DataFrame, out_dir: str) -> str:
    fig, axes = plt.subplots(4, 1, figsize=(15, 18), dpi=130,
                              gridspec_kw={"hspace": 0.55})
    fig.suptitle("3GPP TS 28.552 §5.2.6 – PDU Session Management KPIs",
                 fontsize=13, fontweight="bold")
    df = df.copy(); df["timestamp"] = pd.to_datetime(df["timestamp"])
    rule = "1h"

    ax = axes[0]
    for i, inst in enumerate(sorted(df["amf_instance_id"].unique())):
        sub = (df[df["amf_instance_id"]==inst]
               .set_index("timestamp")[["SM.PduSessEstabAtt","SM.PduSessEstabSucc"]]
               .resample(rule).sum())
        ax.plot(sub.index, sub["SM.PduSessEstabAtt"], color=_PALETTE[i%8], lw=1.2, label=f"{inst} Att")
        ax.plot(sub.index, sub["SM.PduSessEstabSucc"], color=_PALETTE[i%8], lw=0.8, ls="--", alpha=0.7)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "SM.PduSessEstabAtt / Succ (solid=Att, dashed=Succ)", "PDU Sessions / h")
    ax.legend(ncol=5, fontsize=7)

    ax = axes[1]
    sr = _resample_agg(df, ["SM.PduSessEstabSuccRate"], "mean", rule)
    ax.plot(sr.index, sr["SM.PduSessEstabSuccRate"], color="#d62728", lw=1.5)
    ax.axhline(99.0, color="orange", ls="--", lw=1.0, label="99 % ref")
    ax.set_ylim(85, 101)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "SM.PduSessEstabSuccRate (%)", "Success Rate (%)")
    ax.legend(fontsize=8)

    ax = axes[2]
    rm = _resample_agg(df, ["SM.PduSessRelAtt","SM.PduSessRelSucc",
                              "SM.PduSessModAtt","SM.PduSessModSucc"], "sum", rule)
    for col, lbl, c in zip(["SM.PduSessRelAtt","SM.PduSessRelSucc","SM.PduSessModAtt","SM.PduSessModSucc"],
                             ["RelAtt","RelSucc","ModAtt","ModSucc"],
                             ["#1f77b4","#aec7e8","#ff7f0e","#ffbb78"]):
        if col in rm.columns:
            ax.plot(rm.index, rm[col], color=c, lw=1.1, label=lbl)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "SM – PDU Session Release & Modification Att/Succ", "Count / h")
    ax.legend(ncol=4, fontsize=7)

    ax = axes[3]
    vas = _resample_agg(df, ["SM.VoNRAtt","SM.EPSFallbackAtt","SM.SMSAtt"], "sum", rule)
    for col, lbl, c in zip(["SM.VoNRAtt","SM.EPSFallbackAtt","SM.SMSAtt"],
                             ["VoNR","EPS Fallback Voice","SMS"],
                             ["#2ca02c","#d62728","#9467bd"]):
        if col in vas.columns:
            ax.plot(vas.index, vas[col], color=c, lw=1.2, label=lbl)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "SM – Value-Added Services (VoNR / EPS Fallback / SMS) Attempts", "Count / h")
    ax.legend(fontsize=8)

    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "sec526_pdu_session.png")
    fig.savefig(path, bbox_inches="tight", dpi=130); plt.close(fig)
    print(f"  §5.2.6  → {path}"); return path


# ── Section 5.2.7 ──────────────────────────────────────────────────────────


### `def plot_527_authentication`

In [24]:
def plot_527_authentication(df: pd.DataFrame, out_dir: str) -> str:
    fig, axes = plt.subplots(3, 1, figsize=(15, 13), dpi=130,
                              gridspec_kw={"hspace": 0.55})
    fig.suptitle("3GPP TS 28.552 §5.2.7 – Authentication & NAS Security KPIs",
                 fontsize=13, fontweight="bold")
    df = df.copy(); df["timestamp"] = pd.to_datetime(df["timestamp"])
    rule = "1h"

    ax = axes[0]
    auth = _resample_agg(df, ["AUTH.AuthProcAtt","AUTH.AuthProcSucc","AUTH.AuthProcFail"], "sum", rule)
    ax.fill_between(auth.index, auth.get("AUTH.AuthProcAtt", pd.Series(0, index=auth.index)),
                    alpha=0.25, color="#1f77b4", label="AuthProcAtt")
    ax.plot(auth.index, auth.get("AUTH.AuthProcSucc", pd.Series(0, index=auth.index)),
            color="#2ca02c", lw=1.3, label="AuthProcSucc")
    ax.plot(auth.index, auth.get("AUTH.AuthProcFail", pd.Series(0, index=auth.index)),
            color="#d62728", lw=1.1, ls="--", label="AuthProcFail")
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "AUTH – Authentication Procedures Att / Succ / Fail", "Auth Procs / h")
    ax.legend(fontsize=8)

    ax = axes[1]
    sr = _resample_agg(df, ["AUTH.AuthSuccRate"], "mean", rule)
    ax.plot(sr.index, sr["AUTH.AuthSuccRate"], color="#d62728", lw=1.5)
    ax.axhline(99.5, color="orange", ls="--", lw=1.0, label="99.5 % ref")
    ax.set_ylim(85, 101)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "AUTH.AuthSuccRate (%)", "Success Rate (%)")
    ax.legend(fontsize=8)

    ax = axes[2]
    nas = _resample_agg(df, ["AUTH.NasSecModeAtt","AUTH.NasSecModeSucc"], "sum", rule)
    ax.fill_between(nas.index, nas.get("AUTH.NasSecModeAtt", pd.Series(0, index=nas.index)),
                    alpha=0.3, color="#9467bd", label="NasSecModeAtt")
    ax.plot(nas.index, nas.get("AUTH.NasSecModeSucc", pd.Series(0, index=nas.index)),
            color="#8c564b", lw=1.2, label="NasSecModeSucc")
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "AUTH – NAS Security Mode Command Att / Succ", "NAS-SMC / h")
    ax.legend(fontsize=8)

    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "sec527_authentication.png")
    fig.savefig(path, bbox_inches="tight", dpi=130); plt.close(fig)
    print(f"  §5.2.7  → {path}"); return path


# ── Section 5.2.8 ──────────────────────────────────────────────────────────


### `def plot_528_n1n2`

In [25]:
def plot_528_n1n2(df: pd.DataFrame, out_dir: str) -> str:
    fig, axes = plt.subplots(3, 1, figsize=(15, 13), dpi=130,
                              gridspec_kw={"hspace": 0.55})
    fig.suptitle("3GPP TS 28.552 §5.2.8 – N1/N2 Interface Load KPIs",
                 fontsize=13, fontweight="bold")
    df = df.copy(); df["timestamp"] = pd.to_datetime(df["timestamp"])
    rule = "1h"

    ax = axes[0]
    n1 = _resample_agg(df, ["N1N2.N1MsgSent","N1N2.N1MsgRecv"], "sum", rule)
    ax.plot(n1.index, n1.get("N1N2.N1MsgSent", pd.Series(0, index=n1.index)),
            color="#1f77b4", lw=1.3, label="N1MsgSent")
    ax.plot(n1.index, n1.get("N1N2.N1MsgRecv", pd.Series(0, index=n1.index)),
            color="#aec7e8", lw=1.0, ls="--", label="N1MsgRecv")
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "N1 Interface Messages Sent / Received", "Messages / h")
    ax.legend(fontsize=8)

    ax = axes[1]; ax2 = ax.twinx()
    n2 = _resample_agg(df, ["N1N2.N2MsgSent","N1N2.N2MsgRecv"], "sum", rule)
    ngap = _resample_agg(df, ["N1N2.NgapConnActive"], "mean", rule)
    ax.plot(n2.index, n2.get("N1N2.N2MsgSent", pd.Series(0, index=n2.index)),
            color="#ff7f0e", lw=1.3, label="N2MsgSent")
    ax.plot(n2.index, n2.get("N1N2.N2MsgRecv", pd.Series(0, index=n2.index)),
            color="#ffbb78", lw=1.0, ls="--", label="N2MsgRecv")
    ax2.plot(ngap.index, ngap.get("N1N2.NgapConnActive", pd.Series(0, index=ngap.index)),
             color="#2ca02c", lw=1.0, label="NgapConnActive")
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "N2 Interface Messages & Active NGAP Connections", "Messages / h")
    ax2.set_ylabel("Active NGAP Conn", fontsize=9)
    ax.legend(loc="upper left", fontsize=8); ax2.legend(loc="upper right", fontsize=8)

    ax = axes[2]
    ml = _resample_agg(df, ["N1N2.TotalMsgLoad"], "sum", rule)
    ax.fill_between(ml.index, ml.get("N1N2.TotalMsgLoad", pd.Series(0, index=ml.index)),
                    alpha=0.4, color="#d62728", label="TotalMsgLoad")
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "N1N2.TotalMsgLoad – Aggregate N1+N2 Message Load", "Total Msgs / h")
    ax.legend(fontsize=8)

    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "sec528_n1n2_interface.png")
    fig.savefig(path, bbox_inches="tight", dpi=130); plt.close(fig)
    print(f"  §5.2.8  → {path}"); return path


# ── Section 5.2.9 ──────────────────────────────────────────────────────────


### `def plot_529_resource`

In [26]:
def plot_529_resource(df: pd.DataFrame, out_dir: str) -> str:
    fig, axes = plt.subplots(5, 1, figsize=(15, 22), dpi=130,
                              gridspec_kw={"hspace": 0.55})
    fig.suptitle("3GPP TS 28.552 §5.2.9 – Resource & Performance KPIs",
                 fontsize=13, fontweight="bold")
    df = df.copy(); df["timestamp"] = pd.to_datetime(df["timestamp"])
    rule = "1h"

    ax = axes[0]
    for i, inst in enumerate(sorted(df["amf_instance_id"].unique())):
        sub = df[df["amf_instance_id"]==inst].set_index("timestamp")[["RES.CpuUtil"]].resample(rule).mean()
        ax.plot(sub.index, sub["RES.CpuUtil"], color=_PALETTE[i%8], lw=1.2, label=inst)
    ax.axhline(80, color="orange", ls="--", lw=1.0, label="80 % warning")
    ax.axhline(95, color="red", ls="--", lw=1.0, label="95 % critical")
    ax.set_ylim(0, 105)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "RES.CpuUtil (%) per AMF Instance", "CPU Util (%)")
    ax.legend(ncol=4, fontsize=7)

    ax = axes[1]
    for i, inst in enumerate(sorted(df["amf_instance_id"].unique())):
        sub = df[df["amf_instance_id"]==inst].set_index("timestamp")[["RES.MemUtil"]].resample(rule).mean()
        ax.plot(sub.index, sub["RES.MemUtil"], color=_PALETTE[i%8], lw=1.2, label=inst)
    ax.axhline(85, color="orange", ls="--", lw=1.0, label="85 % warning")
    ax.set_ylim(0, 105)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "RES.MemUtil (%) per AMF Instance", "Memory Util (%)")
    ax.legend(ncol=4, fontsize=7)

    ax = axes[2]
    for i, inst in enumerate(sorted(df["amf_instance_id"].unique())):
        sub = df[df["amf_instance_id"]==inst].set_index("timestamp")[["RES.Latency_ms"]].resample(rule).mean()
        ax.plot(sub.index, sub["RES.Latency_ms"], color=_PALETTE[i%8], lw=1.2, label=inst)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "RES.Latency_ms – Mean NAS Latency per AMF Instance", "Latency (ms)")
    ax.legend(ncol=4, fontsize=7)

    ax = axes[3]; ax2 = ax.twinx()
    tput = _resample_agg(df, ["RES.Throughput_mps"], "mean", rule)
    qlen = _resample_agg(df, ["RES.QueueLength"], "sum", rule)
    ax.plot(tput.index, tput.get("RES.Throughput_mps", pd.Series(0, index=tput.index)),
            color="#1f77b4", lw=1.3, label="Throughput (msgs/s)")
    ax2.plot(qlen.index, qlen.get("RES.QueueLength", pd.Series(0, index=qlen.index)),
             color="#d62728", lw=0.9, ls="--", label="QueueLength (sum)")
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "RES.Throughput_mps & RES.QueueLength", "Msgs/s")
    ax2.set_ylabel("Queue Length", fontsize=9)
    ax.legend(loc="upper left", fontsize=8); ax2.legend(loc="upper right", fontsize=8)

    ax = axes[4]
    ue_agg = _resample_agg(df, ["RES.ActiveUEs","RES.ConnectedUEs","RES.IdleUEs"], "sum", rule)
    for col, lbl, c in zip(["RES.ActiveUEs","RES.ConnectedUEs","RES.IdleUEs"],
                             ["Active UEs","CM-Connected","Idle"],
                             ["#2ca02c","#1f77b4","#ff7f0e"]):
        if col in ue_agg.columns:
            ax.plot(ue_agg.index, ue_agg[col], color=c, lw=1.2, label=lbl)
    _shade_anomalies(ax, None, df)
    _setup_ax(ax, "RES – Active / Connected / Idle UE Counts (all AMF summed)", "UE Count")
    ax.legend(fontsize=8)

    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "sec529_resource_performance.png")
    fig.savefig(path, bbox_inches="tight", dpi=130); plt.close(fig)
    print(f"  §5.2.9  → {path}"); return path


# ── Correlation analysis ────────────────────────────────────────────────────


### `def plot_correlations`

In [27]:
def plot_correlations(df: pd.DataFrame, out_dir: str) -> str:
    os.makedirs(out_dir, exist_ok=True)
    df = df.copy(); df["timestamp"] = pd.to_datetime(df["timestamp"])
    CORR_MAP = {
        "RES.CpuUtil":"CPU Util %","RES.MemUtil":"Mem Util %",
        "RES.Latency_ms":"Latency (ms)","RES.Wq_ms":"Queue Wait (ms)",
        "RES.QueueLength":"Queue Length","RES.Throughput_mps":"Throughput (m/s)",
        "UC.ActiveUeContext":"Active UE Ctx","RM.RegReqAtt":"Reg Attempts",
        "CM.ServiceReqAtt":"Svc Req Att","MM.HoReqAtt":"HO Attempts",
        "SM.PduSessEstabAtt":"PDU Estab Att","AUTH.AuthProcAtt":"Auth Attempts",
        "N1N2.TotalMsgLoad":"N1/N2 Msg Load","composite_load":"Composite Load",
    }
    avail = {k: v for k, v in CORR_MAP.items() if k in df.columns}
    corr_df = df[list(avail.keys())].rename(columns=avail).dropna()
    corr_mat = corr_df.corr(method="pearson")

    _ANOM_COLOR = {"none":"#aec6e8","cpu_overload":"#d62728",
                   "ddos_fake_registrations":"#ff7f0e","memory_leak":"#9467bd",
                   "registration_storm":"#8c564b","handover_failure":"#e377c2",
                   "signaling_storm":"#17becf","slice_isolation_failure":"#e53935",
                   "amf_overload_cascade":"#4a148c"}

    fig = plt.figure(figsize=(20, 18), dpi=130)
    fig.suptitle("AMF KPI Correlation Analysis – v14", fontsize=15, fontweight="bold", y=0.98)
    gs = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, :])
    im = ax1.imshow(corr_mat.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax1.set_xticks(range(len(corr_mat.columns)))
    ax1.set_yticks(range(len(corr_mat.columns)))
    ax1.set_xticklabels(corr_mat.columns, rotation=40, ha="right", fontsize=8)
    ax1.set_yticklabels(corr_mat.columns, fontsize=8)
    for i in range(len(corr_mat)):
        for j in range(len(corr_mat.columns)):
            v = corr_mat.values[i, j]
            ax1.text(j, i, f"{v:+.2f}", ha="center", va="center", fontsize=7,
                     color="white" if abs(v) > 0.6 else "black",
                     fontweight="bold" if abs(v) > 0.7 else "normal")
    fig.colorbar(im, ax=ax1, shrink=0.6, pad=0.01).set_label("Pearson r", fontsize=9)
    ax1.set_title("Pearson Correlation Matrix – Resource × Traffic × Latency KPIs",
                  fontsize=11, fontweight="bold", pad=10)

    ax2 = fig.add_subplot(gs[1, 0])
    if "RES.CpuUtil" in df.columns and "RES.Latency_ms" in df.columns:
        for atype in df["anomaly_type"].unique():
            sub = df[df["anomaly_type"] == atype]
            ax2.scatter(sub["RES.CpuUtil"], sub["RES.Latency_ms"],
                        c=_ANOM_COLOR.get(atype, "#aec6e8"), alpha=0.3, s=5,
                        label=atype if atype != "none" else "Normal", rasterized=True)
        cpu_v = df["RES.CpuUtil"].values; lat_v = df["RES.Latency_ms"].values
        msk = np.isfinite(cpu_v) & np.isfinite(lat_v)
        if msk.sum() > 10:
            z = np.polyfit(cpu_v[msk], lat_v[msk], 2)
            xf = np.linspace(cpu_v[msk].min(), cpu_v[msk].max(), 200)
            ax2.plot(xf, np.polyval(z, xf), "k--", lw=1.5, label="Quadratic fit", zorder=5)
        r = np.corrcoef(cpu_v[msk], lat_v[msk])[0, 1]
        ax2.set_xlabel("CPU Util (%)", fontsize=9)
        ax2.set_ylabel("NAS Latency (ms)", fontsize=9)
        ax2.set_title(f"CPU Util vs NAS Latency  (r = {r:+.3f})", fontsize=10, fontweight="bold")
        ax2.legend(fontsize=6, ncol=2, loc="upper left"); ax2.grid(True, alpha=0.3)

    ax3 = fig.add_subplot(gs[1, 1])
    if "RES.MemUtil" in df.columns and "UC.ActiveUeContext" in df.columns:
        for atype in df["anomaly_type"].unique():
            sub = df[df["anomaly_type"] == atype]
            ax3.scatter(sub["UC.ActiveUeContext"], sub["RES.MemUtil"],
                        c=_ANOM_COLOR.get(atype, "#aec6e8"), alpha=0.3, s=5,
                        label=atype if atype != "none" else "Normal", rasterized=True)
        ctx_v = df["UC.ActiveUeContext"].values; mem_v = df["RES.MemUtil"].values
        msk2 = np.isfinite(ctx_v) & np.isfinite(mem_v)
        if msk2.sum() > 10:
            z2 = np.polyfit(ctx_v[msk2], mem_v[msk2], 1)
            xf2 = np.linspace(ctx_v[msk2].min(), ctx_v[msk2].max(), 200)
            ax3.plot(xf2, np.polyval(z2, xf2), "k--", lw=1.5, label="Linear fit", zorder=5)
        r2 = np.corrcoef(ctx_v[msk2], mem_v[msk2])[0, 1]
        ax3.set_xlabel("Active UE Contexts", fontsize=9)
        ax3.set_ylabel("Memory Util (%)", fontsize=9)
        ax3.set_title(f"Active UE Contexts vs Memory Util  (r = {r2:+.3f})", fontsize=10, fontweight="bold")
        ax3.legend(fontsize=6, ncol=2, loc="upper left"); ax3.grid(True, alpha=0.3)

    path = os.path.join(out_dir, "correlation_analysis.png")
    fig.savefig(path, bbox_inches="tight", dpi=130); plt.close(fig)
    print(f"  Correlation → {path}"); return path


# ── Per-procedure latency comparison ───────────────────────────────────────


### `def plot_procedure_latency`

In [28]:
def plot_procedure_latency(df: pd.DataFrame, out_dir: str) -> str:
    os.makedirs(out_dir, exist_ok=True)
    lat_cols = sorted([c for c in df.columns if c.startswith("RES.Lat_") and "_ms" in c
                       and c not in ("RES.Lat_eMBB_ms","RES.Lat_mMTC_ms","RES.Lat_URLLC_ms")])
    if not lat_cols:
        return ""
    proc_labels = {
        "RES.Lat_InitReg_ms":"Initial Reg","RES.Lat_MobReg_ms":"Mobility Reg",
        "RES.Lat_PerReg_ms":"Periodic Reg","RES.Lat_SrvReq_ms":"Service Req",
        "RES.Lat_N2Rel_ms":"N2 Release","RES.Lat_XnHO_ms":"Xn Handover",
        "RES.Lat_N2HO_ms":"N2 Handover","RES.Lat_PduEstab_ms":"PDU Estab",
        "RES.Lat_Auth_ms":"Auth (AUSF)","RES.Lat_Paging_ms":"Paging",
    }
    labels  = [proc_labels.get(c, c) for c in lat_cols]
    normal  = df[df["is_anomaly"] == 0]
    anomaly = df[df["is_anomaly"] == 1]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=130)
    fig.suptitle("Per-Procedure NAS Latency – v14", fontsize=13, fontweight="bold")

    x = np.arange(len(lat_cols)); w = 0.35
    ax = axes[0]
    bp1 = ax.boxplot([normal[c].dropna().values  for c in lat_cols],
                     positions=x - w/2, widths=w*0.8, patch_artist=True,
                     medianprops=dict(color="black", linewidth=1.5))
    bp2 = ax.boxplot([anomaly[c].dropna().values for c in lat_cols] if not anomaly.empty else
                     [np.array([0]) for _ in lat_cols],
                     positions=x + w/2, widths=w*0.8, patch_artist=True,
                     medianprops=dict(color="black", linewidth=1.5))
    for p in bp1["boxes"]: p.set_facecolor("#aec6e8")
    for p in bp2["boxes"]: p.set_facecolor("#f4a261")
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
    ax.set_ylabel("Latency (ms)"); ax.set_title("Normal vs Anomaly periods")
    ax.legend([bp1["boxes"][0], bp2["boxes"][0]], ["Normal","Anomaly"], fontsize=9)
    ax.grid(axis="y", alpha=0.3)

    ax2 = axes[1]
    means = {proc_labels.get(c, c): df[c].mean() for c in lat_cols}
    means_s = dict(sorted(means.items(), key=lambda kv: kv[1], reverse=True))
    colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, len(means_s)))
    bars = ax2.barh(list(means_s.keys()), list(means_s.values()), color=colors, alpha=0.85, edgecolor="white")
    ax2.bar_label(bars, fmt="%.1f ms", padding=3, fontsize=8)
    ax2.set_xlabel("Mean Latency (ms)"); ax2.set_title("Mean NAS Latency per Procedure Type")
    ax2.invert_yaxis(); ax2.grid(axis="x", alpha=0.3)

    plt.tight_layout()
    path = os.path.join(out_dir, "procedure_latency.png")
    fig.savefig(path, bbox_inches="tight", dpi=130); plt.close(fig)
    print(f"  Procedure latency → {path}"); return path


# ── Memory breakdown ────────────────────────────────────────────────────────


### `def plot_memory_breakdown`

In [29]:
def plot_memory_breakdown(df: pd.DataFrame, out_dir: str) -> str:
    os.makedirs(out_dir, exist_ok=True)
    mem_components = {
        "RES.MemBase_MB":"Base OS + AMF","RES.MemCtx_MB":"UE Contexts",
        "RES.MemPDU_MB":"PDU Anchors","RES.MemSigBuf_MB":"Sig Buffers",
        "RES.MemAuthCache_MB":"Auth Cache",
    }
    available = {k: v for k, v in mem_components.items() if k in df.columns}
    if not available:
        return ""
    mem_cols  = list(available.keys())
    inst_id   = df["amf_instance_id"].unique()[0]
    inst0 = (df[df["amf_instance_id"]==inst_id][["timestamp"] + mem_cols]
             .copy().set_index("timestamp").sort_index()
             .resample("1h").mean(numeric_only=True))
    colors_mem = ["#4878d0","#ee854a","#6acc65","#d65f5f","#956cb4"]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5), dpi=130)
    fig.suptitle("AMF Memory Breakdown – v14 (Chiha et al. 2020 calibration)",
                 fontsize=12, fontweight="bold")

    ax = axes[0]; bot = np.zeros(len(inst0))
    for (col, lbl), c in zip(available.items(), colors_mem):
        if col in inst0.columns:
            vals = inst0[col].fillna(0).values
            ax.fill_between(inst0.index, bot, bot + vals, label=lbl, alpha=0.8, color=c)
            bot += vals
    ax.set_ylabel("Memory (MB)"); ax.set_title(f"Memory components over time ({inst_id})")
    ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=0.3)

    ax2 = axes[1]
    avg_vals = [df[c].mean() for c in available]
    ax2.pie(avg_vals, labels=[f"{v}\n({x:.0f} MB)" for v, x in zip(available.values(), avg_vals)],
            colors=colors_mem[:len(avg_vals)], autopct="%1.1f%%", startangle=140,
            textprops={"fontsize": 8})
    ax2.set_title(f"Average memory breakdown\n(total: {sum(avg_vals):.0f} MB avg / {MEM_MAX_MB:.0f} MB max)")

    plt.tight_layout()
    path = os.path.join(out_dir, "memory_breakdown.png")
    fig.savefig(path, bbox_inches="tight", dpi=130); plt.close(fig)
    print(f"  Memory breakdown → {path}"); return path


# ── Master dispatcher for all TS 28.552 section plots ──────────────────────


### `def plot_all_sections`

In [30]:
def plot_all_sections(df: pd.DataFrame, out_dir: str) -> List[str]:
    """Generate one dedicated figure per 3GPP TS 28.552 §5.2.x section."""
    os.makedirs(out_dir, exist_ok=True)
    paths = []
    for fn in [plot_521_registration, plot_522_connection, plot_523_mobility,
               plot_524_paging, plot_525_ue_context, plot_526_pdu_session,
               plot_527_authentication, plot_528_n1n2, plot_529_resource,
               plot_correlations, plot_procedure_latency, plot_memory_breakdown]:
        try:
            p = fn(df, out_dir)
            if p:
                paths.append(p)
        except Exception as exc:
            print(f"  [warn] {fn.__name__} failed: {exc}")
    print(f"\n  All section plots saved to: {out_dir}")
    return paths


### `def plot_overview`

In [31]:
def plot_overview(df: pd.DataFrame, out_dir: str) -> str:
    os.makedirs(out_dir, exist_ok=True)
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    agg = df.groupby("timestamp").agg(
        reg_att =("RM.RegReqAtt",     "sum"),
        reg_sr  =("RM.RegSuccRate",   "mean"),
        ho_att  =("MM.HoReqAtt",      "sum"),
        ho_sr   =("MM.HoSuccRate",    "mean"),
        pag_att =("PAG.PagingReqAtt", "sum"),
        ctx_ut  =("UC.ContextUtilRate","mean"),
        cpu     =("RES.CpuUtil",      "mean"),
        mem     =("RES.MemUtil",      "mean"),
        lat     =("RES.Latency_ms",   "mean"),
    ).reset_index()

    fig, axes = plt.subplots(3, 2, figsize=(18, 14), dpi=120)
    fig.suptitle("Synthetic AMF KPI Dataset – v14 Overview (3GPP TS 28.552)",
                 fontsize=14, fontweight="bold", y=1.01)

    ax = axes[0, 0]; ax2 = ax.twinx()
    ax.bar(agg["timestamp"], agg["reg_att"], width=0.01, alpha=0.4, color="steelblue", label="RegReqAtt")
    ax2.plot(agg["timestamp"], agg["reg_sr"], color="red", lw=1.3, label="RegSuccRate (%)")
    _setup_ax(ax, "5.2.1 Registration Management")
    ax.set_ylabel("Attempts"); ax2.set_ylabel("Success Rate (%)")
    ax.legend(loc="upper left", fontsize=8); ax2.legend(loc="upper right", fontsize=8)

    ax = axes[0, 1]; ax2 = ax.twinx()
    ax.bar(agg["timestamp"], agg["ho_att"], width=0.01, alpha=0.4, color="darkorange", label="HoReqAtt")
    ax2.plot(agg["timestamp"], agg["ho_sr"], color="purple", lw=1.3, label="HoSuccRate (%)")
    _setup_ax(ax, "5.2.3 Mobility Management (Handover)")
    ax.set_ylabel("Attempts"); ax2.set_ylabel("Success Rate (%)")
    ax.legend(loc="upper left", fontsize=8); ax2.legend(loc="upper right", fontsize=8)

    ax = axes[1, 0]
    for inst in sorted(df["amf_instance_id"].unique()):
        g = df[df["amf_instance_id"] == inst].set_index("timestamp").resample("1h")["RES.CpuUtil"].mean()
        ax.plot(g.index, g.values, lw=1.1, label=inst, alpha=0.85)
    _setup_ax(ax, "5.2.9 CPU Utilisation per AMF Instance")
    ax.set_ylabel("CPU Util (%)"); ax.set_ylim(0, 100); ax.legend(fontsize=8, ncol=2)

    ax = axes[1, 1]
    for inst in sorted(df["amf_instance_id"].unique()):
        vals = df.loc[(df["amf_instance_id"] == inst) & (df["RES.Latency_ms"] > 0), "RES.Latency_ms"].dropna()
        if len(vals) > 10:
            ax.hist(vals, bins=80, density=True, alpha=0.4, label=inst)
    ax.set_xscale("log")
    _setup_ax(ax, "5.2.9 Latency PDF (Log-Normal, log scale)")
    ax.set_xlabel("Latency (ms)"); ax.set_ylabel("PDF"); ax.legend(fontsize=8, ncol=2)

    ax = axes[2, 0]; ax2 = ax.twinx()
    ax.fill_between(agg["timestamp"], agg["pag_att"], alpha=0.35, color="teal", label="PagingReqAtt")
    ax2.plot(agg["timestamp"], agg["ctx_ut"], color="brown", lw=1.2, label="ContextUtilRate (%)")
    _setup_ax(ax, "5.2.4 Paging + 5.2.5 UE Context Utilisation")
    ax.set_ylabel("Paging Att"); ax2.set_ylabel("Ctx Util (%)");
    ax.legend(loc="upper left", fontsize=8); ax2.legend(loc="upper right", fontsize=8)

    ax = axes[2, 1]
    pivot    = df.pivot_table(index="amf_instance_id", columns="timestamp",
                               values="is_anomaly", aggfunc="max")
    pivot_rs = pivot.T.resample("1h").max().T
    im = ax.imshow(pivot_rs.values, aspect="auto", cmap="Reds", vmin=0, vmax=1)
    ax.set_yticks(range(pivot_rs.shape[0])); ax.set_yticklabels(pivot_rs.index, fontsize=9)
    ticks = np.linspace(0, pivot_rs.shape[1]-1, 8).astype(int)
    ax.set_xticks(ticks)
    ax.set_xticklabels([pivot_rs.columns[i].strftime("%m/%d") for i in ticks], rotation=30, fontsize=8)
    ax.set_title("Anomaly Timeline Heatmap (hourly)", fontweight="bold", fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.03, label="Anomaly Active")

    plt.tight_layout()
    path = os.path.join(out_dir, "amf_kpi_overview_v14.png")
    fig.savefig(path, bbox_inches="tight", dpi=120)
    plt.close(fig)
    print(f"  Overview → {path}")
    return path


### `def plot_per_slice_resources`

In [32]:
def plot_per_slice_resources(df: pd.DataFrame, out_dir: str) -> str:
    """v14 NEW: Per-slice CPU, Memory, and Latency comparison plots."""
    os.makedirs(out_dir, exist_ok=True)
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    agg = df.groupby("timestamp").agg(
        cpu_embb  =("RES.CPU_eMBB_pct",  "mean"),
        cpu_mmtc  =("RES.CPU_mMTC_pct",  "mean"),
        cpu_urllc =("RES.CPU_URLLC_pct", "mean"),
        mem_embb  =("RES.Mem_eMBB_MB",   "mean"),
        mem_mmtc  =("RES.Mem_mMTC_MB",   "mean"),
        mem_urllc =("RES.Mem_URLLC_MB",  "mean"),
        lat_embb  =("RES.Lat_eMBB_ms",   "mean"),
        lat_mmtc  =("RES.Lat_mMTC_ms",   "mean"),
        lat_urllc =("RES.Lat_URLLC_ms",  "mean"),
        sla_embb  =("SLA.eMBB_breach",   "mean"),
        sla_mmtc  =("SLA.mMTC_breach",   "mean"),
        sla_urllc =("SLA.URLLC_breach",  "mean"),
    ).reset_index()

    fig, axes = plt.subplots(2, 2, figsize=(16, 10), dpi=120)
    fig.suptitle("v14 Per-Slice Resource KPIs – eMBB / mMTC / URLLC",
                 fontsize=13, fontweight="bold")

    # CPU
    ax = axes[0, 0]
    for cls, col, c in [("eMBB","cpu_embb",SLICE_COLORS["eMBB"]),
                         ("mMTC","cpu_mmtc",SLICE_COLORS["mMTC"]),
                         ("URLLC","cpu_urllc",SLICE_COLORS["URLLC"])]:
        g = agg.set_index("timestamp")[col].resample("1h").mean()
        ax.plot(g.index, g.values, color=c, lw=1.2, label=cls)
    _setup_ax(ax, "Per-Slice CPU Utilisation (%)")
    ax.set_ylabel("CPU %"); ax.set_ylim(0, 100); ax.legend(fontsize=9)

    # Memory
    ax = axes[0, 1]
    for cls, col, c in [("eMBB","mem_embb",SLICE_COLORS["eMBB"]),
                         ("mMTC","mem_mmtc",SLICE_COLORS["mMTC"]),
                         ("URLLC","mem_urllc",SLICE_COLORS["URLLC"])]:
        g = agg.set_index("timestamp")[col].resample("1h").mean()
        ax.plot(g.index, g.values, color=c, lw=1.2, label=cls)
    _setup_ax(ax, "Per-Slice Memory Footprint (MB)")
    ax.set_ylabel("Memory (MB)"); ax.legend(fontsize=9)

    # Latency
    ax = axes[1, 0]
    for cls, col, c, sla in [
            ("eMBB",  "lat_embb",  SLICE_COLORS["eMBB"],  SLICE_LAT_PARAMS["eMBB"]["sla_ms"]),
            ("mMTC",  "lat_mmtc",  SLICE_COLORS["mMTC"],  SLICE_LAT_PARAMS["mMTC"]["sla_ms"]),
            ("URLLC", "lat_urllc", SLICE_COLORS["URLLC"], SLICE_LAT_PARAMS["URLLC"]["sla_ms"]),
    ]:
        g = agg.set_index("timestamp")[col].resample("1h").mean()
        ax.plot(g.index, g.values, color=c, lw=1.2, label=f"{cls} (SLA {sla} ms)")
        ax.axhline(sla, color=c, lw=0.8, ls="--", alpha=0.6)
    _setup_ax(ax, "Per-Slice NAS Latency (ms) with SLA Thresholds")
    ax.set_ylabel("Latency (ms)"); ax.set_yscale("log"); ax.legend(fontsize=8)

    # SLA breach rate
    ax = axes[1, 1]
    for cls, col, c in [("eMBB","sla_embb",SLICE_COLORS["eMBB"]),
                         ("mMTC","sla_mmtc",SLICE_COLORS["mMTC"]),
                         ("URLLC","sla_urllc",SLICE_COLORS["URLLC"])]:
        g = agg.set_index("timestamp")[col].resample("4h").mean() * 100.0
        ax.fill_between(g.index, g.values, color=c, alpha=0.4, label=cls)
    _setup_ax(ax, "SLA Breach Rate per Slice (% of slots)")
    ax.set_ylabel("Breach Rate (%)"); ax.set_ylim(0, 100); ax.legend(fontsize=9)

    plt.tight_layout()
    path = os.path.join(out_dir, "amf_per_slice_resources_v14.png")
    fig.savefig(path, bbox_inches="tight", dpi=120)
    plt.close(fig)
    print(f"  Per-slice resources → {path}")
    return path


### `def plot_anomaly_sigmoid`

In [33]:
def plot_anomaly_sigmoid(df: pd.DataFrame, out_dir: str) -> str:
    """v14 NEW: Visualise sigmoid onset/recovery of anomaly intensity."""
    os.makedirs(out_dir, exist_ok=True)
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    anom_df = df[df["is_anomaly"] == 1].copy()
    if anom_df.empty:
        return ""

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120)
    fig.suptitle("v14 Sigmoid Anomaly Onset / Recovery", fontsize=12, fontweight="bold")

    ax = axes[0]
    for inst in sorted(df["amf_instance_id"].unique()):
        sub = df[df["amf_instance_id"] == inst].set_index("timestamp")
        ax.plot(sub.index, sub["anomaly_sigmoid_w"], lw=0.8, label=inst, alpha=0.75)
    _setup_ax(ax, "Sigmoid Weight per Instance (0=normal, 1=full anomaly)")
    ax.set_ylabel("Sigmoid Weight"); ax.set_ylim(-0.05, 1.05); ax.legend(fontsize=7, ncol=2)

    ax = axes[1]
    sub0 = df[df["amf_instance_id"] == "AMF_00"].set_index("timestamp")
    ax2  = ax.twinx()
    ax.plot(sub0.index, sub0["RES.CpuUtil"],    color="steelblue", lw=1.0, label="CPU %")
    ax2.plot(sub0.index, sub0["anomaly_sigmoid_w"], color="red", lw=0.8, ls="--", label="Sigmoid W")
    _setup_ax(ax, "AMF_00: CPU vs Sigmoid Anomaly Weight")
    ax.set_ylabel("CPU (%)"); ax2.set_ylabel("Sigmoid W")
    ax.legend(loc="upper left", fontsize=8); ax2.legend(loc="upper right", fontsize=8)

    plt.tight_layout()
    path = os.path.join(out_dir, "amf_anomaly_sigmoid_v14.png")
    fig.savefig(path, bbox_inches="tight", dpi=120)
    plt.close(fig)
    print(f"  Anomaly sigmoid → {path}")
    return path


### `def plot_statistical_validation`

In [34]:
def plot_statistical_validation(df: pd.DataFrame, out_dir: str) -> str:
    os.makedirs(out_dir, exist_ok=True)
    inst0 = df[df["amf_instance_id"] == "AMF_00"].copy().set_index("timestamp")

    fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=110)
    fig.suptitle("Statistical Validation – AMF_00", fontsize=12, fontweight="bold")

    # ACF
    ax = axes[0, 0]
    ser = inst0["RM.RegReqAtt"].dropna()
    n   = len(ser)
    nlags = min(96, n // 4)
    acf_vals = [ser.autocorr(lag=l) for l in range(1, nlags + 1)]
    ax.bar(range(1, nlags + 1), acf_vals, width=0.8, color="steelblue", alpha=0.7)
    ax.axhline(1.96 / np.sqrt(n), ls="--", color="red", lw=0.9)
    ax.axhline(-1.96 / np.sqrt(n), ls="--", color="red", lw=0.9)
    ax.set_title("ACF – RM.RegReqAtt (LRD / fGn signature)", fontweight="bold", fontsize=10)
    ax.set_xlabel("Lag (15-min slots)"); ax.set_ylabel("ACF")
    ax.grid(True, alpha=0.3)

    # QQ latency
    ax = axes[0, 1]
    lat = inst0["RES.Latency_ms"].dropna()
    lat = lat[lat > 0]
    log_lat = np.log(lat)
    from scipy.stats import probplot
    (quantiles, values), (slope, intercept, r) = probplot(log_lat, dist="norm")
    ax.scatter(quantiles, values, s=3, alpha=0.4, color="steelblue")
    ax.plot(quantiles, slope * quantiles + intercept,
            color="red", lw=1.2, ls="--")
    ax.set_title("Q-Q: log(Latency_ms) vs Normal\n(validates Log-Normal assumption)",
                 fontweight="bold", fontsize=10)
    ax.set_xlabel("Theoretical Normal Quantiles"); ax.set_ylabel("log Latency")
    ax.grid(True, alpha=0.3)

    # Diurnal box
    ax = axes[1, 0]
    inst0_r = inst0.reset_index()
    inst0_r["hour"] = pd.to_datetime(inst0_r["timestamp"]).dt.hour
    boxes   = [inst0_r[inst0_r["hour"] == h]["RM.RegReqAtt"].values for h in range(24)]
    bp      = ax.boxplot(boxes, positions=range(24), widths=0.6,
                         patch_artist=True, showfliers=False)
    for patch in bp["boxes"]:
        patch.set_facecolor("#AED6F1")
    ax.set_title("Diurnal Pattern: RegReqAtt by Hour-of-Day", fontweight="bold", fontsize=10)
    ax.set_xlabel("Hour"); ax.set_ylabel("RegReqAtt"); ax.grid(True, alpha=0.3)

    # Per-slice latency CDFs
    ax = axes[1, 1]
    for cls, col, c in [("eMBB",  "RES.Lat_eMBB_ms",  SLICE_COLORS["eMBB"]),
                         ("mMTC",  "RES.Lat_mMTC_ms",  SLICE_COLORS["mMTC"]),
                         ("URLLC", "RES.Lat_URLLC_ms", SLICE_COLORS["URLLC"])]:
        vals = inst0[col].dropna().sort_values()
        cdf  = np.arange(1, len(vals)+1) / len(vals)
        ax.plot(vals, cdf, color=c, lw=1.5, label=cls)
        sla = SLICE_LAT_PARAMS[cls]["sla_ms"]
        ax.axvline(sla, color=c, ls="--", lw=0.8, alpha=0.7)
    ax.set_xscale("log")
    ax.set_title("Per-Slice Latency CDF (dashed = SLA threshold)", fontweight="bold", fontsize=10)
    ax.set_xlabel("Latency (ms)"); ax.set_ylabel("CDF")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(out_dir, "amf_stat_validation_v14.png")
    fig.savefig(path, bbox_inches="tight", dpi=110)
    plt.close(fig)
    print(f"  Statistical validation → {path}")
    return path


---
## Section 10 — Plotting — Correlation · Latency · Memory · Per-Slice (v14 new)

In [35]:
# ── Section 10 plotting functions ─────────────────────────────────────────

def plot_correlations(df, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    df = df.copy(); df["timestamp"] = pd.to_datetime(df["timestamp"])
    CORR_MAP = {
        "RES.CpuUtil":"CPU Util %","RES.MemUtil":"Mem Util %",
        "RES.Latency_ms":"Latency (ms)","RES.Wq_ms":"Queue Wait (ms)",
        "RES.QueueLength":"Queue Length","RES.Throughput_mps":"Throughput (m/s)",
        "UC.ActiveUeContext":"Active UE Ctx","RM.RegReqAtt":"Reg Attempts",
        "CM.ServiceReqAtt":"Svc Req Att","MM.HoReqAtt":"HO Attempts",
        "SM.PduSessEstabAtt":"PDU Estab Att","AUTH.AuthProcAtt":"Auth Attempts",
        "N1N2.TotalMsgLoad":"N1/N2 Msg Load",
    }
    avail    = {k:v for k,v in CORR_MAP.items() if k in df.columns}
    corr_mat = df[list(avail.keys())].rename(columns=avail).dropna().corr(method="pearson")
    _ANOM_COLOR = {"none":"#aec6e8","cpu_overload":"#d62728",
                   "ddos_fake_registrations":"#ff7f0e","memory_leak":"#9467bd",
                   "registration_storm":"#8c564b","handover_failure":"#e377c2",
                   "signaling_storm":"#17becf","slice_isolation_failure":"#e53935",
                   "amf_overload_cascade":"#4a148c"}
    fig = plt.figure(figsize=(20,18), dpi=130)
    fig.suptitle("AMF KPI Correlation Analysis – v14", fontsize=15, fontweight="bold", y=0.98)
    gs  = fig.add_gridspec(2,2,hspace=0.42,wspace=0.35)
    ax1 = fig.add_subplot(gs[0,:])
    im  = ax1.imshow(corr_mat.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax1.set_xticks(range(len(corr_mat.columns))); ax1.set_yticks(range(len(corr_mat.columns)))
    ax1.set_xticklabels(corr_mat.columns, rotation=40, ha="right", fontsize=8)
    ax1.set_yticklabels(corr_mat.columns, fontsize=8)
    for i in range(len(corr_mat)):
        for j in range(len(corr_mat.columns)):
            v = corr_mat.values[i,j]
            ax1.text(j,i,f"{v:+.2f}",ha="center",va="center",fontsize=7,
                     color="white" if abs(v)>0.6 else "black",
                     fontweight="bold" if abs(v)>0.7 else "normal")
    fig.colorbar(im,ax=ax1,shrink=0.6,pad=0.01).set_label("Pearson r",fontsize=9)
    ax1.set_title("Pearson Correlation Matrix – Resource × Traffic × Latency KPIs",
                  fontsize=11,fontweight="bold",pad=10)
    ax2 = fig.add_subplot(gs[1,0])
    if "RES.CpuUtil" in df.columns and "RES.Latency_ms" in df.columns:
        for atype in df["anomaly_type"].unique():
            sub = df[df["anomaly_type"]==atype]
            ax2.scatter(sub["RES.CpuUtil"],sub["RES.Latency_ms"],
                        c=_ANOM_COLOR.get(atype,"#aec6e8"),alpha=0.3,s=5,
                        label=atype if atype!="none" else "Normal",rasterized=True)
        cpu_v=df["RES.CpuUtil"].values; lat_v=df["RES.Latency_ms"].values
        msk=np.isfinite(cpu_v)&np.isfinite(lat_v)
        if msk.sum()>10:
            z=np.polyfit(cpu_v[msk],lat_v[msk],2)
            xf=np.linspace(cpu_v[msk].min(),cpu_v[msk].max(),200)
            ax2.plot(xf,np.polyval(z,xf),"k--",lw=1.5,label="Quadratic fit",zorder=5)
        r=np.corrcoef(cpu_v[msk],lat_v[msk])[0,1]
        ax2.set_xlabel("CPU Util (%)"); ax2.set_ylabel("NAS Latency (ms)")
        ax2.set_title(f"CPU Util vs NAS Latency  (r={r:+.3f})",fontsize=10,fontweight="bold")
        ax2.legend(fontsize=6,ncol=2,loc="upper left"); ax2.grid(True,alpha=0.3)
    ax3 = fig.add_subplot(gs[1,1])
    if "RES.MemUtil" in df.columns and "UC.ActiveUeContext" in df.columns:
        for atype in df["anomaly_type"].unique():
            sub = df[df["anomaly_type"]==atype]
            ax3.scatter(sub["UC.ActiveUeContext"],sub["RES.MemUtil"],
                        c=_ANOM_COLOR.get(atype,"#aec6e8"),alpha=0.3,s=5,
                        label=atype if atype!="none" else "Normal",rasterized=True)
        ctx_v=df["UC.ActiveUeContext"].values; mem_v=df["RES.MemUtil"].values
        msk2=np.isfinite(ctx_v)&np.isfinite(mem_v)
        if msk2.sum()>10:
            z2=np.polyfit(ctx_v[msk2],mem_v[msk2],1)
            xf2=np.linspace(ctx_v[msk2].min(),ctx_v[msk2].max(),200)
            ax3.plot(xf2,np.polyval(z2,xf2),"k--",lw=1.5,label="Linear fit",zorder=5)
        r2=np.corrcoef(ctx_v[msk2],mem_v[msk2])[0,1]
        ax3.set_xlabel("Active UE Contexts"); ax3.set_ylabel("Memory Util (%)")
        ax3.set_title(f"Active UE Contexts vs Memory  (r={r2:+.3f})",fontsize=10,fontweight="bold")
        ax3.legend(fontsize=6,ncol=2,loc="upper left"); ax3.grid(True,alpha=0.3)
    path = os.path.join(_PLOT_DIR,"correlation_analysis.png")
    fig.savefig(path,bbox_inches="tight",dpi=130); plt.close(fig)
    print(f"  Correlation → {path}"); return path


def plot_procedure_latency(df, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    lat_cols = sorted([c for c in df.columns if c.startswith("RES.Lat_") and "_ms" in c
                       and c not in ("RES.Lat_eMBB_ms","RES.Lat_mMTC_ms","RES.Lat_URLLC_ms")])
    if not lat_cols: return ""
    proc_labels = {
        "RES.Lat_InitReg_ms":"Initial Reg","RES.Lat_MobReg_ms":"Mobility Reg",
        "RES.Lat_PerReg_ms":"Periodic Reg","RES.Lat_SrvReq_ms":"Service Req",
        "RES.Lat_N2Rel_ms":"N2 Release","RES.Lat_XnHO_ms":"Xn Handover",
        "RES.Lat_N2HO_ms":"N2 Handover","RES.Lat_PduEstab_ms":"PDU Estab",
        "RES.Lat_Auth_ms":"Auth (AUSF)","RES.Lat_Paging_ms":"Paging",
    }
    labels  = [proc_labels.get(c,c) for c in lat_cols]
    normal  = df[df["is_anomaly"]==0]; anomaly = df[df["is_anomaly"]==1]
    fig,axes = plt.subplots(1,2,figsize=(16,6),dpi=130)
    fig.suptitle("Per-Procedure NAS Latency – v14",fontsize=13,fontweight="bold")
    x=np.arange(len(lat_cols)); w=0.35
    bp1=axes[0].boxplot([normal[c].dropna().values for c in lat_cols],
                        positions=x-w/2,widths=w*0.8,patch_artist=True,
                        medianprops=dict(color="black",linewidth=1.5))
    bp2=axes[0].boxplot([anomaly[c].dropna().values if not anomaly.empty else np.array([0])
                         for c in lat_cols],
                        positions=x+w/2,widths=w*0.8,patch_artist=True,
                        medianprops=dict(color="black",linewidth=1.5))
    for p in bp1["boxes"]: p.set_facecolor("#aec6e8")
    for p in bp2["boxes"]: p.set_facecolor("#f4a261")
    axes[0].set_xticks(x); axes[0].set_xticklabels(labels,rotation=35,ha="right",fontsize=8)
    axes[0].set_ylabel("Latency (ms)"); axes[0].set_title("Normal vs Anomaly periods")
    axes[0].legend([bp1["boxes"][0],bp2["boxes"][0]],["Normal","Anomaly"],fontsize=9)
    axes[0].grid(axis="y",alpha=0.3)
    means   = {proc_labels.get(c,c):df[c].mean() for c in lat_cols}
    means_s = dict(sorted(means.items(),key=lambda kv:kv[1],reverse=True))
    colors  = plt.cm.RdYlGn_r(np.linspace(0.15,0.85,len(means_s)))
    bars    = axes[1].barh(list(means_s.keys()),list(means_s.values()),color=colors,alpha=0.85,edgecolor="white")
    axes[1].bar_label(bars,fmt="%.1f ms",padding=3,fontsize=8)
    axes[1].set_xlabel("Mean Latency (ms)"); axes[1].set_title("Mean NAS Latency per Procedure")
    axes[1].invert_yaxis(); axes[1].grid(axis="x",alpha=0.3)
    plt.tight_layout()
    path=os.path.join(out_dir,"procedure_latency.png")
    fig.savefig(path,bbox_inches="tight",dpi=130); plt.close(fig)
    print(f"  Procedure latency → {path}"); return path


def plot_memory_breakdown(df, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    mem_components = {
        "RES.MemBase_MB":"Base OS + AMF","RES.MemCtx_MB":"UE Contexts",
        "RES.MemPDU_MB":"PDU Anchors","RES.MemSigBuf_MB":"Sig Buffers",
        "RES.MemAuthCache_MB":"Auth Cache",
    }
    available = {k:v for k,v in mem_components.items() if k in df.columns}
    if not available: return ""
    inst_id = df["amf_instance_id"].unique()[0]
    inst0   = (df[df["amf_instance_id"]==inst_id][["timestamp"]+list(available.keys())]
               .assign(timestamp=lambda x: pd.to_datetime(x["timestamp"]))
               .set_index("timestamp").sort_index()
               .resample("1h").mean(numeric_only=True))
    colors_mem = ["#4878d0","#ee854a","#6acc65","#d65f5f","#956cb4"]
    fig,axes = plt.subplots(1,2,figsize=(15,5),dpi=130)
    fig.suptitle("AMF Memory Breakdown – v14",fontsize=12,fontweight="bold")
    bot=np.zeros(len(inst0))
    for (col,lbl),c in zip(available.items(),colors_mem):
        if col in inst0.columns:
            vals=inst0[col].fillna(0).values
            axes[0].fill_between(inst0.index,bot,bot+vals,label=lbl,alpha=0.8,color=c)
            bot+=vals
    axes[0].set_ylabel("Memory (MB)"); axes[0].set_title(f"Memory components over time ({inst_id})")
    axes[0].legend(fontsize=8,loc="upper left"); axes[0].grid(alpha=0.3)
    avg_vals=[df[c].mean() for c in available]
    axes[1].pie(avg_vals,labels=[f"{v}\n({x:.0f} MB)" for v,x in zip(available.values(),avg_vals)],
                colors=colors_mem[:len(avg_vals)],autopct="%1.1f%%",startangle=140,
                textprops={"fontsize":8})
    axes[1].set_title(f"Average memory breakdown\n(total: {sum(avg_vals):.0f} MB avg)")
    plt.tight_layout()
    path=os.path.join(out_dir,"memory_breakdown.png")
    fig.savefig(path,bbox_inches="tight",dpi=130); plt.close(fig)
    print(f"  Memory breakdown → {path}"); return path


def plot_per_slice_resources(df, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    df = df.copy(); df["timestamp"] = pd.to_datetime(df["timestamp"])
    agg = df.groupby("timestamp").agg(
        cpu_embb  =("RES.CPU_eMBB_pct",  "mean"),
        cpu_mmtc  =("RES.CPU_mMTC_pct",  "mean"),
        cpu_urllc =("RES.CPU_URLLC_pct", "mean"),
        mem_embb  =("RES.Mem_eMBB_MB",   "mean"),
        mem_mmtc  =("RES.Mem_mMTC_MB",   "mean"),
        mem_urllc =("RES.Mem_URLLC_MB",  "mean"),
        lat_embb  =("RES.Lat_eMBB_ms",   "mean"),
        lat_mmtc  =("RES.Lat_mMTC_ms",   "mean"),
        lat_urllc =("RES.Lat_URLLC_ms",  "mean"),
        sla_embb  =("SLA.eMBB_breach",   "mean"),
        sla_mmtc  =("SLA.mMTC_breach",   "mean"),
        sla_urllc =("SLA.URLLC_breach",  "mean"),
    ).reset_index()
    SC = {"eMBB":"#1f77b4","mMTC":"#2ca02c","URLLC":"#d62728"}
    SLA= {"eMBB":100.0,"mMTC":6000.0,"URLLC":5.0}
    fig,axes = plt.subplots(2,2,figsize=(16,10),dpi=120)
    fig.suptitle("v14 Per-Slice Resource KPIs – eMBB / mMTC / URLLC",fontsize=13,fontweight="bold")
    for cls,col in [("eMBB","cpu_embb"),("mMTC","cpu_mmtc"),("URLLC","cpu_urllc")]:
        g=agg.set_index("timestamp")[col].resample("1h").mean()
        axes[0,0].plot(g.index,g.values,color=SC[cls],lw=1.2,label=cls)
    axes[0,0].set_ylabel("CPU %"); axes[0,0].set_ylim(0,100)
    axes[0,0].set_title("Per-Slice CPU Utilisation (%)"); axes[0,0].legend(fontsize=9); axes[0,0].grid(alpha=0.3)
    for cls,col in [("eMBB","mem_embb"),("mMTC","mem_mmtc"),("URLLC","mem_urllc")]:
        g=agg.set_index("timestamp")[col].resample("1h").mean()
        axes[0,1].plot(g.index,g.values,color=SC[cls],lw=1.2,label=cls)
    axes[0,1].set_ylabel("Memory (MB)"); axes[0,1].set_title("Per-Slice Memory Footprint (MB)")
    axes[0,1].legend(fontsize=9); axes[0,1].grid(alpha=0.3)
    for cls,col in [("eMBB","lat_embb"),("mMTC","lat_mmtc"),("URLLC","lat_urllc")]:
        g=agg.set_index("timestamp")[col].resample("1h").mean()
        axes[1,0].plot(g.index,g.values,color=SC[cls],lw=1.2,label=f"{cls} (SLA {SLA[cls]:.0f}ms)")
        axes[1,0].axhline(SLA[cls],color=SC[cls],lw=0.8,ls="--",alpha=0.6)
    axes[1,0].set_ylabel("Latency (ms)"); axes[1,0].set_yscale("log")
    axes[1,0].set_title("Per-Slice NAS Latency with SLA Thresholds"); axes[1,0].legend(fontsize=8); axes[1,0].grid(alpha=0.3)
    for cls,col in [("eMBB","sla_embb"),("mMTC","sla_mmtc"),("URLLC","sla_urllc")]:
        g=agg.set_index("timestamp")[col].resample("4h").mean()*100.0
        axes[1,1].fill_between(g.index,g.values,color=SC[cls],alpha=0.4,label=cls)
    axes[1,1].set_ylabel("Breach Rate (%)"); axes[1,1].set_ylim(0,100)
    axes[1,1].set_title("SLA Breach Rate per Slice (% of slots)"); axes[1,1].legend(fontsize=9); axes[1,1].grid(alpha=0.3)
    plt.tight_layout()
    path=os.path.join(out_dir,"amf_per_slice_resources_v14.png")
    fig.savefig(path,bbox_inches="tight",dpi=120); plt.close(fig)
    print(f"  Per-slice resources → {path}"); return path


print("Section 10 functions ready — run the 'Generate All Plots' cell to produce the figures.")


Section 10 functions ready — run the 'Generate All Plots' cell to produce the figures.


---
## Section 11 — Configuration Parameters


In [36]:
# =============================================================================
# Section 11 — Configuration Parameters
# Edit the values below, then run this cell followed by "Generate Dataset".
# =============================================================================

# ── Topology ─────────────────────────────────────────────────────────────────
AMF_INSTANCES_RUN   = 1          # Released benchmark: 1. Framework supports up to 5.
RANDOM_SEED_RUN     = 42         # Master RNG seed — change to reproduce a different run.

# Duration options:
#   336   → 14 days  (ablation studies, rapid iteration)
#   720   → 30 days
#   1440  → 60 days  (released benchmark, paper results)  ← DEFAULT
#   2160  → 90 days
DURATION_HOURS_RUN  = 1440

# ── Network Slices (UE counts) ───────────────────────────────────────────────
UE_EMBB_RUN         = 70_000    # eMBB UEs   (paper benchmark: 70,000)
UE_MMTC_RUN         = 20_000    # mMTC UEs   (paper benchmark: 20,000)
UE_URLLC_RUN        = 10_000    # URLLC UEs  (paper benchmark: 10,000)

# ── Hardware Resources per AMF Instance ──────────────────────────────────────
VCPUS_RUN           = 8          # vCPU cores per AMF instance
MEM_MAX_MB_RUN      = 8192       # RAM per AMF instance (MB)
#   Options: 4096 (small lab), 8192 (default cloud pod),
#            16384 (mid-tier K8s), 32768 (large node), 65536 (carrier-grade)

# ── Measurement Granularity ──────────────────────────────────────────────────
STEP_MIN_RUN        = 15         # Slot duration in minutes.
#   Options: 1, 5, 15 (3GPP standard default), 30, 60

# ── Anomaly Injection ────────────────────────────────────────────────────────
INJECT_ANOMALIES    = True       # Set False to produce a clean baseline dataset.
ANOMALY_INTENSITY   = "moderate" # "mild" | "moderate" | "severe"

# ─────────────────────────────────────────────────────────────────────────────
# Derived summary (printed on run — do not edit below this line)
# ─────────────────────────────────────────────────────────────────────────────
_total_ues = UE_EMBB_RUN + UE_MMTC_RUN + UE_URLLC_RUN
_days      = DURATION_HOURS_RUN // 24
_rows      = (DURATION_HOURS_RUN * 60 // STEP_MIN_RUN) * AMF_INSTANCES_RUN

print("=" * 60)
print(f"  AMF instances  : {AMF_INSTANCES_RUN}")
print(f"  Random seed    : {RANDOM_SEED_RUN}")
print(f"  Duration       : {DURATION_HOURS_RUN} h  ({_days} days)")
print(f"  Step           : {STEP_MIN_RUN} min  →  {_rows:,} expected rows")
print(f"  UEs            : {_total_ues:,}  "
      f"(eMBB {UE_EMBB_RUN:,} / mMTC {UE_MMTC_RUN:,} / URLLC {UE_URLLC_RUN:,})")
print(f"  vCPUs / RAM    : {VCPUS_RUN}  /  {MEM_MAX_MB_RUN} MB")
print(f"  Anomalies      : {'YES @ ' + ANOMALY_INTENSITY if INJECT_ANOMALIES else 'NO (clean dataset)'}")
print("=" * 60)


  AMF instances  : 1
  Random seed    : 42
  Duration       : 1440 h  (60 days)
  Step           : 15 min  →  5,760 expected rows
  UEs            : 100,000  (eMBB 70,000 / mMTC 20,000 / URLLC 10,000)
  vCPUs / RAM    : 8  /  8192 MB
  Anomalies      : YES @ moderate


---
## Cell — Generate Dataset

In [37]:
# =============================================================================
# Generate Dataset
# Reads configuration from Section 11 parameters cell.
# =============================================================================

# Build scenario list
if INJECT_ANOMALIES:
    _scenarios = [dict(sc, intensity=ANOMALY_INTENSITY) for sc in ANOMALY_SCENARIOS]
else:
    _scenarios = []

print(f"Anomalies : {'YES – ' + str(len(_scenarios)) + ' scenarios @ ' + ANOMALY_INTENSITY if _scenarios else 'NO (clean dataset)'}")

gen = AMFDatasetGenerator(
    seed              = RANDOM_SEED_RUN,
    amf_instances     = AMF_INSTANCES_RUN,
    ue_embb           = UE_EMBB_RUN,
    ue_mmtc           = UE_MMTC_RUN,
    ue_urllc          = UE_URLLC_RUN,
    include_anomalies = INJECT_ANOMALIES,
    anomaly_scenarios = _scenarios,
    vcpus_per_amf     = VCPUS_RUN,
    mem_max_mb        = MEM_MAX_MB_RUN,
    duration_hours    = DURATION_HOURS_RUN,
    step_min          = STEP_MIN_RUN,
)
df = gen.generate()

print(f"\n✓ Shape (internal) : {df.shape}")
print(f"  AMF instances    : {df['amf_instance_id'].nunique()}")
print(f"  Anomalous rows   : {df['is_anomaly'].sum()} / {len(df)}")
print(f"  SLA breaches     → eMBB: {df['SLA.eMBB_breach'].sum()}  "
      f"mMTC: {df['SLA.mMTC_breach'].sum()}  URLLC: {df['SLA.URLLC_breach'].sum()}")
print(f"\nPreview (first 3 rows of internal DataFrame):")
display(df.head(3))


Anomalies : YES – 24 scenarios @ moderate

✓ Shape (internal) : (5760, 116)
  AMF instances    : 1
  Anomalous rows   : 162 / 5760
  SLA breaches     → eMBB: 0  mMTC: 0  URLLC: 0

Preview (first 3 rows of internal DataFrame):


,timestamp,amf_instance_id,is_anomaly,anomaly_type,anomaly_intensity,anomaly_sigmoid_w,composite_load,rho,RM.RegReqAtt,RM.RegReqSucc,...,RES.Lat_URLLC_ms,RES.CpuCyclesPerMsg,RES.MemBytesPerUE,RES.SliceLoadEntropy,RES.Jackson_eMBB_eps,RES.Jackson_mMTC_eps,RES.Jackson_URLLC_eps,SLA.eMBB_breach,SLA.mMTC_breach,SLA.URLLC_breach
0,2024-01-01 00:00:00,AMF_00,0,none,none,0.0,0.2422,0.0397,14038,14020,...,0.413,214950.2,30816.5,1.2503,310.931,71.212,45.583,0,0,0
1,2024-01-01 00:15:00,AMF_00,0,none,none,0.0,0.3049,0.0452,19599,19556,...,0.606,204111.8,32239.5,1.2243,391.952,90.685,52.184,0,0,0
2,2024-01-01 00:30:00,AMF_00,0,none,none,0.0,0.2355,0.0355,14803,14743,...,0.461,173197.3,33768.1,1.2130,338.407,65.933,30.387,0,0,0


---
## Cell — Validate Dataset

In [38]:
_expected_rows = (DURATION_HOURS_RUN * 60 // STEP_MIN_RUN) * AMF_INSTANCES_RUN
v = validate_dataset(df, amf_instances=AMF_INSTANCES_RUN)
v['row_count_correct'] = len(df) == _expected_rows
print('Validation results:')
for k, ok in v.items():
    print(f"  {'\u2713' if ok else '\u2717'} {k}")
print('\n\u2713 All checks passed.' if all(v.values()) else '\n\u26a0  Some checks failed.')


Validation results:
  ✓ row_count_correct
  ✓ no_nan_in_key_cols
  ✓ rates_in_range
  ✓ counts_non_negative
  ✓ anomaly_labels_ok
  ✓ correct_amf_instances
  ✓ timestamps_ordered
  ✓ cpu_in_range
  ✓ mem_in_range
  ✓ succ_le_att

✓ All checks passed.


---
## Cell — Statistics Summary

In [39]:
_stat_cols = [
    'RES.CpuUtil','RES.MemUtil','RES.Latency_ms',
    'RES.Lat_eMBB_ms','RES.Lat_mMTC_ms','RES.Lat_URLLC_ms',
    'RES.CPU_eMBB_pct','RES.CPU_mMTC_pct','RES.CPU_URLLC_pct',
    'RES.CpuCyclesPerMsg','RES.MemBytesPerUE','RES.SliceLoadEntropy',
]
summary = df[[c for c in _stat_cols if c in df.columns]].describe(
    percentiles=[.50,.90,.95,.99]).T
from IPython.display import display, HTML
display(HTML(f'<b>📊 KPI Statistics (full {DURATION_HOURS_RUN//24}-day window, {AMF_INSTANCES_RUN} instance(s))</b>'))
display(summary.style.format('{:.2f}').background_gradient(cmap='Blues', axis=0))


,count,mean,std,min,50%,90%,95%,99%,max
RES.CpuUtil,5760.00,72.77,27.40,15.37,82.18,99.50,99.50,99.50,99.50
RES.MemUtil,5760.00,11.92,0.94,8.99,11.92,13.03,13.36,14.48,16.94
RES.Latency_ms,5760.00,9.57,1.34,5.61,9.44,11.24,11.85,13.71,19.32
RES.Lat_eMBB_ms,5760.00,1.01,0.30,0.31,0.96,1.41,1.58,1.90,3.33
RES.Lat_mMTC_ms,5760.00,2.00,1.03,0.29,1.79,3.28,3.92,5.65,10.31
RES.Lat_URLLC_ms,5760.00,0.50,0.08,0.28,0.50,0.61,0.65,0.73,1.04
RES.CPU_eMBB_pct,5760.00,79.79,26.94,12.72,98.66,99.50,99.50,99.50,99.50
RES.CPU_mMTC_pct,5760.00,4.01,1.61,0.50,3.95,6.00,6.60,7.97,18.08
RES.CPU_URLLC_pct,5760.00,14.72,7.77,0.50,14.12,25.52,27.32,30.82,66.11
RES.CpuCyclesPerMsg,5760.00,192464.28,25064.02,114405.80,190389.55,225482.40,236296.89,259808.87,319621.10


---
## Cell — Generate All Plots
Produces the full plot suite:
- **Overview dashboard** (registration, handover, CPU/mem, latency PDF, anomaly heatmap)
- **Per-section** §5.2.1 – §5.2.9 (one figure per KPI section)
- **Per-slice resources** (eMBB / mMTC / URLLC CPU, memory, latency, SLA breach)
- **Sigmoid anomaly envelope** (v14 new)
- **Statistical validation** (ACF, Q-Q, diurnal box, per-slice CDF)
- **Correlation heatmap** + scatter plots
- **Per-procedure latency** (normal vs anomaly box-plots)
- **Memory breakdown** (5-component stacked area)

In [40]:
import os, glob
import matplotlib.image as mpimg

_PLOT_DIR = '/content/amf_plots_v14'
os.makedirs(_PLOT_DIR, exist_ok=True)

print('[1/5] Overview dashboard ...')
plot_overview(df, _PLOT_DIR)

print('[2/5] Per-section §5.2.1 – §5.2.9 ...')
_sec_dir = os.path.join(_PLOT_DIR, 'per_section')
os.makedirs(_sec_dir, exist_ok=True)
plot_all_sections(df, _sec_dir)

print('[3/5] Per-slice resources + anomaly sigmoid ...')
plot_per_slice_resources(df, _PLOT_DIR)
plot_anomaly_sigmoid(df, _PLOT_DIR)

print('[4/5] Statistical validation ...')
plot_statistical_validation(df, _PLOT_DIR)

print('[5/5] Correlation + procedure latency + memory breakdown ...')
plot_correlations(df, _PLOT_DIR)
plot_procedure_latency(df, _PLOT_DIR)
plot_memory_breakdown(df, _PLOT_DIR)

print(f'\n✓ All plots saved to: {_PLOT_DIR}')

# Show first 4 plots inline
for _p in sorted(glob.glob(os.path.join(_PLOT_DIR, '*.png')))[:4]:
    _img = mpimg.imread(_p)
    _fig, _ax = plt.subplots(1, 1, figsize=(14, 8))
    _ax.imshow(_img); _ax.axis('off')
    _ax.set_title(os.path.basename(_p), fontsize=9)
    plt.tight_layout(); plt.show()


[1/5] Overview dashboard ...
  Overview → /content/amf_plots_v14/amf_kpi_overview_v14.png
[2/5] Per-section §5.2.1 – §5.2.9 ...
  §5.2.1  → /content/amf_plots_v14/per_section/sec521_registration_management.png
  §5.2.2  → /content/amf_plots_v14/per_section/sec522_connection_management.png
  §5.2.3  → /content/amf_plots_v14/per_section/sec523_mobility_management.png
  §5.2.4  → /content/amf_plots_v14/per_section/sec524_paging.png
  §5.2.5  → /content/amf_plots_v14/per_section/sec525_ue_context.png
  §5.2.6  → /content/amf_plots_v14/per_section/sec526_pdu_session.png
  §5.2.7  → /content/amf_plots_v14/per_section/sec527_authentication.png
  §5.2.8  → /content/amf_plots_v14/per_section/sec528_n1n2_interface.png
  §5.2.9  → /content/amf_plots_v14/per_section/sec529_resource_performance.png
  Correlation → /content/amf_plots_v14/correlation_analysis.png
  Procedure latency → /content/amf_plots_v14/per_section/procedure_latency.png
  Memory breakdown → /content/amf_plots_v14/per_section/memo

---
## Cell — Export & Download

In [41]:
# =============================================================================
# Export & Download
# Strips leakage columns, writes public 112-column artifact, builds ZIP.
# =============================================================================
import os, zipfile, json as _json
from datetime import datetime

# ── Output directory (single consistent path) ─────────────────────────────
_OUT  = "/content/amf_dataset_v14"
_PLOT_DIR = "/content/amf_plots_v14"
os.makedirs(_OUT, exist_ok=True)

_csv  = os.path.join(_OUT, "amf_synthetic_dataset_v14.csv")
_jsn  = os.path.join(_OUT, "amf_synthetic_dataset_v14.json")
_meta = os.path.join(_OUT, "metadata_v14.json")
_zip  = os.path.join(_OUT, "amf_synthetic_dataset_v14.zip")

# ── Strip leakage columns to produce the public artifact ─────────────────
public_df, dropped_cols = build_public_release(df)

print(f"Internal DataFrame : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Dropped (leakage)  : {dropped_cols}")
print(f"Public artifact    : {public_df.shape[0]} rows × {public_df.shape[1]} columns")
assert public_df.shape[1] == 112, (
    f"Expected 112 public columns, got {public_df.shape[1]}. "
    "Check PUBLIC_DROP_COLUMNS definition."
)

# ── Write public CSV and JSON ─────────────────────────────────────────────
public_df.to_csv(_csv, index=False)
public_df.to_json(_jsn, orient="records", date_format="iso", indent=2)

# ── Write metadata sidecar ────────────────────────────────────────────────
_meta_dict = {
    "generator":                "amf_synthetic_dataset_v14_colab",
    "3gpp_standard":            "TS 28.552 v19.6.0",
    "generated_at":             datetime.utcnow().isoformat(),
    "random_seed":              RANDOM_SEED_RUN,
    "amf_instances":            AMF_INSTANCES_RUN,
    "duration_hours":           DURATION_HOURS_RUN,
    "step_min":                 STEP_MIN_RUN,
    "ue_embb":                  UE_EMBB_RUN,
    "ue_mmtc":                  UE_MMTC_RUN,
    "ue_urllc":                 UE_URLLC_RUN,
    "total_ues":                UE_EMBB_RUN + UE_MMTC_RUN + UE_URLLC_RUN,
    "vcpus_per_amf":            VCPUS_RUN,
    "mem_max_mb":               MEM_MAX_MB_RUN,
    "anomaly_injection":        INJECT_ANOMALIES,
    "anomaly_intensity":        ANOMALY_INTENSITY if INJECT_ANOMALIES else "none",
    "internal_shape":           {"rows": int(df.shape[0]), "columns": int(df.shape[1])},
    "public_shape":             {"rows": int(public_df.shape[0]), "columns": int(public_df.shape[1])},
    "dropped_leakage_columns":  dropped_cols,
    "total_rows":               int(public_df.shape[0]),
    "columns":                  int(public_df.shape[1]),
    "column_names":             list(public_df.columns),
    "anomaly_rows":             int(df["is_anomaly"].sum()),
    "normal_rows":              int((df["is_anomaly"] == 0).sum()),
    "sla_breaches": {
        "eMBB":  int(public_df["SLA.eMBB_breach"].sum()),
        "mMTC":  int(public_df["SLA.mMTC_breach"].sum()),
        "URLLC": int(public_df["SLA.URLLC_breach"].sum()),
    },
    "statistical_models": [
        "Negative Binomial (count overdispersion, NB-k per procedure)",
        "Log-Normal (latency, per-slice CV)",
        "Erlang-C M/M/c (queueing sojourn time)",
        "Fractional Gaussian Noise (H=0.75, long-range dependence)",
        "GARCH(1,1) (volatility clustering)",
        "Markov chain 3-state UE model (Idle / Connected / Deregistered)",
        "Sigmoid onset/recovery for anomaly intensity ramps",
        "Jackson open-network per-class throughput (BCMP theorem)",
    ],
    "kpi_sections": {
        "5.2.1": "RM.*  Registration Management",
        "5.2.2": "CM.*  Connection Management",
        "5.2.3": "MM.*  Mobility Management",
        "5.2.4": "PAG.* Paging",
        "5.2.5": "UC.*  UE Context",
        "5.2.6": "SM.*  PDU Session",
        "5.2.7": "AUTH.* Authentication / Security",
        "5.2.8": "N1N2.* Interface Load",
        "5.2.9": "RES.* Resource / Performance (+ per-slice v14)",
        "v14_new": "SLA.* breach flags, RES.*_eMBB/mMTC/URLLC",
    },
}
with open(_meta, "w") as fh:
    _json.dump(_meta_dict, fh, indent=2)

# ── Pack public artifact into ZIP ─────────────────────────────────────────
with zipfile.ZipFile(_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in [_csv, _jsn, _meta]:
        zf.write(p, os.path.basename(p))
    if os.path.isdir(_PLOT_DIR):
        for root, _, files in os.walk(_PLOT_DIR):
            for fn in files:
                fp = os.path.join(root, fn)
                zf.write(fp, os.path.relpath(fp, "/content"))

print(f"\n✓ CSV (public)  : {_csv}  ({os.path.getsize(_csv)//1024} KB)")
print(f"✓ JSON (public) : {_jsn}  ({os.path.getsize(_jsn)//1024} KB)")
print(f"✓ Metadata      : {_meta}")
print(f"✓ ZIP           : {_zip}  ({os.path.getsize(_zip)//1024} KB)")

from IPython.display import HTML, display
display(HTML(
    f'<a href="{_zip}" download '
    'style="display:inline-block;padding:10px 22px;background:#1565C0;'
    'color:white;border-radius:5px;text-decoration:none;'
    'font-weight:bold;margin:8px 0">'
    '⬇️  Download ZIP (CSV + JSON + Metadata + Plots)'
    '</a>'
))
try:
    from google.colab import files as _cf
    _cf.download(_zip)
    print("✓ Download triggered via Colab API.")
except Exception:
    print("ℹ  Use the link above or the Files panel to download.")


Internal DataFrame : 5760 rows × 116 columns
Dropped (leakage)  : ['anomaly_sigmoid_w', 'anomaly_intensity', 'composite_load', 'rho']
Public artifact    : 5760 rows × 112 columns


/tmp/ipykernel_4561/2677596117.py:37: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at":             datetime.utcnow().isoformat(),



✓ CSV (public)  : /content/amf_dataset_v14/amf_synthetic_dataset_v14.csv  (3822 KB)
✓ JSON (public) : /content/amf_dataset_v14/amf_synthetic_dataset_v14.json  (19036 KB)
✓ Metadata      : /content/amf_dataset_v14/metadata_v14.json
✓ ZIP           : /content/amf_dataset_v14/amf_synthetic_dataset_v14.zip  (13772 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Download triggered via Colab API.
